# 4D SU(2) validated RG — M1 exact 2D benchmark

This notebook retains the accepted M0 persistence regression suite and is now the user-facing entry point for milestone **M1** only: rigorous rational Wilson coefficients, value/gradient tails, exact 2D RG, and an independent diagonal-convolution verifier. The accepted M0 run is immutable parent evidence and is never resumed or overwritten by M1.

```python
m1_orchestrator.run_until_checkpoint()
```

M2–M6 are deliberately not implemented here. Every path remains fail-closed with `certification_status = 'NOT_CERTIFIED'`. Detailed Paperspace execution and recovery instructions are in the project-root `README.md`.

## Important M1 prerequisites

The accepted M0 checkpoint `/storage/validated_4d_su2_rg/runs/20260719T120406Z-731966c8fd28/checkpoints/ckpt_000014` must remain present and unchanged. Use Python 3.11+, upload the complete bundle, start with a fresh kernel, and run top-to-bottom. M1 uses a new `M1-...` run ID; never set the accepted M0 run ID as the M1 run. See the Japanese `README.md` before execution.

In [ ]:
# pytest is the only package expected to be absent on Paperspace. Install it first.
import importlib.util
import subprocess
import sys

if importlib.util.find_spec('pytest') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pytest>=8'])
import pytest
print('pytest:', pytest.__version__)


In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

_bundle_name = 'validated_4d_su2_rg_codex_bundle'
_explicit_root = os.environ.get('VALIDATED_RG_PROJECT_ROOT')
_candidates = ([Path(_explicit_root)] if _explicit_root else [
    Path.cwd(), Path.cwd().parent, Path.cwd() / _bundle_name, Path.cwd().parent / _bundle_name,
    Path('/notebooks'), Path('/notebooks') / _bundle_name, Path('/storage') / _bundle_name,
])
PROJECT_ROOT = next((
    candidate.expanduser().resolve() for candidate in _candidates
    if (candidate.expanduser() / 'validated_4d_su2_rg_codex_design.md').is_file()
), Path.cwd().resolve())
if not (PROJECT_ROOT / 'validated_4d_su2_rg_codex_design.md').is_file():
    raise RuntimeError('Cannot locate the complete validated_4d_su2_rg_codex_bundle.')
if sys.version_info < (3, 11):
    raise RuntimeError(f'Python 3.11+ is required for M1; found {sys.version.split()[0]}.')
os.environ.setdefault('VALIDATED_RG_PROJECT_ROOT', str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
for _relative in ('src', 'tests'):
    (PROJECT_ROOT / _relative).mkdir(parents=True, exist_ok=True)
print('M1 project root:', PROJECT_ROOT)


In [ ]:
%%writefile src/exact_arithmetic.py
from __future__ import annotations

import json
from dataclasses import dataclass
from decimal import Decimal, ROUND_CEILING, ROUND_FLOOR, localcontext
from fractions import Fraction
from typing import Any


class ExactArithmeticError(ValueError):
    '''Raised when a proof-path interval operation is invalid.'''


def _require_fraction(value: Fraction, name: str) -> None:
    if not isinstance(value, Fraction):
        raise TypeError(f'{name} must be fractions.Fraction, not {type(value)!r}.')


def fraction_payload(value: Fraction) -> dict[str, str]:
    _require_fraction(value, 'value')
    return {
        'numerator_hex': format(value.numerator, 'x'),
        'denominator_hex': format(value.denominator, 'x'),
    }


def fraction_from_payload(payload: dict[str, Any]) -> Fraction:
    if not isinstance(payload, dict):
        raise ExactArithmeticError('Fraction payload must be a mapping.')
    try:
        numerator = int(payload['numerator_hex'], 16)
        denominator = int(payload['denominator_hex'], 16)
    except (KeyError, TypeError, ValueError) as exc:
        raise ExactArithmeticError('Malformed hexadecimal fraction payload.') from exc
    if denominator <= 0:
        raise ExactArithmeticError('Fraction denominator must be positive.')
    return Fraction(numerator, denominator)


def outward_decimal(value: Fraction, places: int, upper: bool) -> str:
    _require_fraction(value, 'value')
    if not isinstance(places, int) or isinstance(places, bool) or places < 1:
        raise ExactArithmeticError('places must be a positive integer.')
    rounding = ROUND_CEILING if upper else ROUND_FLOOR
    with localcontext() as context:
        context.prec = places + 100
        context.rounding = rounding
        quotient = Decimal(value.numerator) / Decimal(value.denominator)
        unit = Decimal(1).scaleb(-places)
        return str(quotient.quantize(unit, rounding=rounding))


@dataclass(frozen=True, slots=True)
class RationalInterval:
    lo: Fraction
    hi: Fraction

    def __post_init__(self) -> None:
        _require_fraction(self.lo, 'lo')
        _require_fraction(self.hi, 'hi')
        if self.lo > self.hi:
            raise ExactArithmeticError('Interval lower endpoint exceeds upper endpoint.')

    @classmethod
    def point(cls, value: Fraction) -> 'RationalInterval':
        _require_fraction(value, 'value')
        return cls(value, value)

    @classmethod
    def from_payload(cls, payload: dict[str, Any]) -> 'RationalInterval':
        if not isinstance(payload, dict):
            raise ExactArithmeticError('Interval payload must be a mapping.')
        return cls(fraction_from_payload(payload['lo']), fraction_from_payload(payload['hi']))

    def to_payload(self, decimal_places: int = 36) -> dict[str, Any]:
        return {
            'lo': fraction_payload(self.lo),
            'hi': fraction_payload(self.hi),
            'decimal_lo': outward_decimal(self.lo, decimal_places, upper=False),
            'decimal_hi': outward_decimal(self.hi, decimal_places, upper=True),
        }

    def __add__(self, other: 'RationalInterval') -> 'RationalInterval':
        if not isinstance(other, RationalInterval):
            raise TypeError('RationalInterval addition requires another RationalInterval.')
        return RationalInterval(self.lo + other.lo, self.hi + other.hi)

    def __sub__(self, other: 'RationalInterval') -> 'RationalInterval':
        if not isinstance(other, RationalInterval):
            raise TypeError('RationalInterval subtraction requires another RationalInterval.')
        return RationalInterval(self.lo - other.hi, self.hi - other.lo)

    def __mul__(self, other: 'RationalInterval') -> 'RationalInterval':
        if not isinstance(other, RationalInterval):
            raise TypeError('RationalInterval multiplication requires another RationalInterval.')
        products = (self.lo * other.lo, self.lo * other.hi, self.hi * other.lo, self.hi * other.hi)
        return RationalInterval(min(products), max(products))

    def scale(self, factor: Fraction) -> 'RationalInterval':
        _require_fraction(factor, 'factor')
        return (
            RationalInterval(self.lo * factor, self.hi * factor)
            if factor >= 0
            else RationalInterval(self.hi * factor, self.lo * factor)
        )

    def divide_positive(self, denominator: 'RationalInterval') -> 'RationalInterval':
        if not isinstance(denominator, RationalInterval) or denominator.lo <= 0:
            raise ExactArithmeticError('Denominator interval must be strictly positive.')
        return self * RationalInterval(Fraction(1, denominator.hi), Fraction(1, denominator.lo))

    def positive_power(self, exponent: int) -> 'RationalInterval':
        if not isinstance(exponent, int) or isinstance(exponent, bool) or exponent < 0:
            raise ExactArithmeticError('Exponent must be a nonnegative integer.')
        if self.lo < 0:
            raise ExactArithmeticError('Positive-power proof path requires a nonnegative interval.')
        return RationalInterval(self.lo**exponent, self.hi**exponent)

    def subset_of(self, outer: 'RationalInterval') -> bool:
        return isinstance(outer, RationalInterval) and outer.lo <= self.lo and self.hi <= outer.hi

    def overlaps(self, other: 'RationalInterval') -> bool:
        return isinstance(other, RationalInterval) and max(self.lo, other.lo) <= min(self.hi, other.hi)

    def assert_nonnegative(self) -> None:
        if self.lo < 0:
            raise ExactArithmeticError('A rigorous nonnegative enclosure has a negative lower endpoint.')


def canonical_json_bytes(payload: Any) -> bytes:
    return json.dumps(payload, sort_keys=True, separators=(',', ':'), ensure_ascii=False, allow_nan=False).encode('utf-8')


In [ ]:
%%writefile src/su2_representations.py
from __future__ import annotations

from dataclasses import dataclass
from fractions import Fraction


@dataclass(frozen=True, order=True, slots=True)
class Irrep:
    '''SU(2) irrep stored exactly as j2 = 2j.'''

    j2: int

    def __post_init__(self) -> None:
        if not isinstance(self.j2, int) or isinstance(self.j2, bool) or self.j2 < 0:
            raise ValueError('j2 must be a nonnegative integer.')

    @property
    def dimension(self) -> int:
        return self.j2 + 1

    @property
    def casimir(self) -> Fraction:
        return Fraction(self.j2 * (self.j2 + 2), 4)

    @property
    def spin(self) -> Fraction:
        return Fraction(self.j2, 2)

    def dual(self) -> 'Irrep':
        return self

    def reverse_orientation(self) -> 'Irrep':
        return self.dual()

    def tensor_product(self, other: 'Irrep') -> tuple['Irrep', ...]:
        if not isinstance(other, Irrep):
            raise TypeError('SU(2) tensor product requires another Irrep.')
        return tuple(Irrep(j2) for j2 in range(abs(self.j2 - other.j2), self.j2 + other.j2 + 1, 2))


CONVENTION = {
    'irrep_coordinate': 'j2=2j nonnegative integer',
    'dimension': 'd_j=j2+1',
    'casimir': 'C2=j2(j2+2)/4=j(j+1)',
    'dual': 'SU(2) irreps are self-dual',
    'class_angle': 'Tr(U)=2 cos(theta)',
    'normalized_weight': 'exp(beta*(cos(theta)-1))',
}


In [ ]:
%%writefile src/su2_special_functions.py
from __future__ import annotations

from fractions import Fraction
from functools import lru_cache
from math import factorial

from .exact_arithmetic import ExactArithmeticError, RationalInterval


def _validate_terms(terms: int) -> None:
    if not isinstance(terms, int) or isinstance(terms, bool) or terms < 1:
        raise ExactArithmeticError('Series terms must be a positive integer.')


@lru_cache(maxsize=4096)
def exp_positive_series_bounds(x: Fraction, terms: int = 120) -> RationalInterval:
    if not isinstance(x, Fraction) or x < 0:
        raise ExactArithmeticError('Exponential proof path requires a nonnegative Fraction.')
    _validate_terms(terms)
    total = Fraction(1)
    term = Fraction(1)
    for k in range(1, terms + 1):
        term *= x / k
        total += term
    next_ratio = x / (terms + 1)
    if next_ratio >= 1:
        raise ExactArithmeticError('Exponential remainder ratio is not contractive; increase terms.')
    remainder_upper = term * next_ratio / (1 - next_ratio)
    return RationalInterval(total, total + remainder_upper)


@lru_cache(maxsize=16384)
def besseli_integer_positive_series_bounds(
    n: int, beta: Fraction, terms: int = 96,
) -> RationalInterval:
    if not isinstance(n, int) or isinstance(n, bool) or n < 0:
        raise ExactArithmeticError('Bessel order must be a nonnegative integer.')
    if not isinstance(beta, Fraction) or beta < 0:
        raise ExactArithmeticError('beta must be a nonnegative Fraction.')
    _validate_terms(terms)
    half_beta = beta / 2
    term = half_beta**n / factorial(n)
    total = term
    for k in range(1, terms + 1):
        term *= half_beta * half_beta / (k * (n + k))
        total += term
    next_ratio = half_beta * half_beta / ((terms + 1) * (n + terms + 1))
    if next_ratio >= 1:
        raise ExactArithmeticError('Bessel remainder ratio is not contractive; increase terms.')
    remainder_upper = term * next_ratio / (1 - next_ratio)
    return RationalInterval(total, total + remainder_upper)


def wilson_character_coefficient_bounds(
    dimension_n: int, beta: Fraction, terms: int = 96,
) -> RationalInterval:
    if not isinstance(dimension_n, int) or isinstance(dimension_n, bool) or dimension_n < 1:
        raise ExactArithmeticError('Irrep dimension n must be a positive integer.')
    if not isinstance(beta, Fraction) or beta <= 0:
        raise ExactArithmeticError('Wilson beta must be a positive Fraction.')
    bessel = besseli_integer_positive_series_bounds(dimension_n, beta, terms)
    return bessel.scale(Fraction(2 * dimension_n, 1) / beta)


def normalized_wilson_coefficient_bounds(
    dimension_n: int, beta: Fraction, terms: int = 96, exp_terms: int = 120,
) -> RationalInterval:
    raw = wilson_character_coefficient_bounds(dimension_n, beta, terms)
    exp_beta = exp_positive_series_bounds(beta, exp_terms)
    return raw.divide_positive(exp_beta)


In [ ]:
%%writefile src/m1_config.py
from __future__ import annotations

import hashlib
import json
import math
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any


@dataclass(frozen=True, slots=True)
class M1Config:
    schema_version: int = 1
    project_name: str = 'validated_4d_su2_rg'
    milestone: str = 'M1'
    parent_milestone: str = 'M0'
    parent_run_id: str = '20260719T120406Z-731966c8fd28'
    parent_checkpoint: str = 'ckpt_000014'
    parent_checkpoint_path: str = '/storage/validated_4d_su2_rg/runs/20260719T120406Z-731966c8fd28/checkpoints/ckpt_000014'
    beta_numerator: int = 11
    beta_denominator: int = 5
    seed: int = 20260719
    cutoffs: tuple[int, ...] = (6, 8, 10, 12, 14, 16)
    dimensions: tuple[int, ...] = (2, 3, 4)
    rg_steps: int = 3
    coefficient_series_terms: int = 96
    exp_series_terms: int = 120
    verifier_series_terms: int = 72
    decimal_places: int = 36
    checkpoint_interval_s: float = 15.0 * 60.0
    max_work_item_s: float = 20.0 * 60.0
    short_task_limit_s: float = 5.0 * 60.0
    checkpoint_reserve_s: float = 10.0 * 60.0
    no_long_task_after_s: float = 5.0 * 3600.0
    drain_after_s: float = 5.0 * 3600.0 + 15.0 * 60.0
    final_save_after_s: float = 5.0 * 3600.0 + 20.0 * 60.0
    hard_return_s: float = 5.0 * 3600.0 + 30.0 * 60.0
    tensor_shard_bytes: int = 256 * 1024 * 1024
    max_item_attempts: int = 3
    certification_status: str = 'NOT_CERTIFIED'

    def __post_init__(self) -> None:
        if self.schema_version != 1 or self.project_name != 'validated_4d_su2_rg' or self.milestone != 'M1':
            raise ValueError('Unsupported M1 project/config schema.')
        if self.parent_milestone != 'M0' or self.certification_status != 'NOT_CERTIFIED':
            raise ValueError('M1 parent/status invariant failed.')
        if not self.parent_run_id or self.parent_checkpoint != 'ckpt_000014':
            raise ValueError('Accepted M0 parent identity is missing or changed.')
        if Path(self.parent_checkpoint_path).name != self.parent_checkpoint:
            raise ValueError('Parent checkpoint path/name mismatch.')
        integers = (
            self.beta_numerator, self.beta_denominator, self.seed, self.rg_steps,
            self.coefficient_series_terms, self.exp_series_terms, self.verifier_series_terms,
            self.decimal_places, self.tensor_shard_bytes, self.max_item_attempts,
        ) + self.cutoffs + self.dimensions
        if any(not isinstance(value, int) or isinstance(value, bool) for value in integers):
            raise ValueError('M1 discrete configuration fields must be integers.')
        if self.beta_numerator <= 0 or self.beta_denominator <= 0 or self.rg_steps < 0:
            raise ValueError('M1 beta and RG step count are invalid.')
        if not 0 <= self.seed < 2**32:
            raise ValueError('M1 seed must satisfy 0 <= seed < 2**32.')
        if tuple(sorted(set(self.cutoffs))) != self.cutoffs or min(self.cutoffs) < 1:
            raise ValueError('M1 cutoffs must be strictly increasing positive integers.')
        if tuple(sorted(set(self.dimensions))) != self.dimensions or min(self.dimensions) < 2:
            raise ValueError('M1 dimensions must be strictly increasing and start above the trivial irrep.')
        if min(self.coefficient_series_terms, self.exp_series_terms, self.verifier_series_terms, self.decimal_places) < 16:
            raise ValueError('M1 rational precision policy is below the fail-closed floor.')
        schedule = (0.0, self.no_long_task_after_s, self.drain_after_s, self.final_save_after_s, self.hard_return_s)
        if tuple(sorted(schedule)) != schedule or len(set(schedule)) != len(schedule):
            raise ValueError('M1 session thresholds must be strictly increasing.')
        durations = schedule[1:] + (self.checkpoint_interval_s, self.max_work_item_s, self.short_task_limit_s, self.checkpoint_reserve_s)
        if any(not isinstance(value, (int, float)) or isinstance(value, bool) or not math.isfinite(value) or value <= 0 for value in durations):
            raise ValueError('M1 duration fields must be positive finite reals.')
        if self.checkpoint_interval_s > 15 * 60 or self.max_work_item_s > 20 * 60:
            raise ValueError('M1 checkpoint/work-item limits exceed the roadmap.')
        if self.final_save_after_s > 5 * 3600 + 20 * 60 or self.hard_return_s > 5.5 * 3600:
            raise ValueError('M1 session finalization exceeds the hard limits.')

    def canonical_payload(self) -> dict[str, Any]:
        payload = asdict(self)
        payload['cutoffs'] = list(self.cutoffs)
        payload['dimensions'] = list(self.dimensions)
        return payload

    def canonical_json(self) -> str:
        return json.dumps(self.canonical_payload(), sort_keys=True, separators=(',', ':'), allow_nan=False)

    def config_hash(self) -> str:
        return hashlib.sha256(self.canonical_json().encode('utf-8')).hexdigest()


In [ ]:
%%writefile src/tail_bounds.py
from __future__ import annotations

from fractions import Fraction
from math import factorial
from typing import Any

from .exact_arithmetic import ExactArithmeticError, RationalInterval
from .su2_special_functions import (
    exp_positive_series_bounds, normalized_wilson_coefficient_bounds,
    wilson_character_coefficient_bounds,
)

VALUE_TAIL_PROOF = (
    'At theta=0, chi_n(I)=n and exp(beta)=sum_{n>=1} n*a_n. '
    'Positivity and |chi_n|<=n give ||tail||_infty <= exp(-beta)*sum_{n>N} n*a_n.'
)
GRADIENT_TAIL_PROOF = (
    'Use the explicit weight-sum bound ||grad chi_j||_infty <= n^2/2, '
    'I_n(beta) <= (beta/2)^n/n!*exp(beta^2/(4(n+1))), and a decreasing term-ratio majorant.'
)


def coefficient_enclosures(
    beta: Fraction, max_dimension: int, series_terms: int, exp_terms: int, decimal_places: int,
) -> dict[str, Any]:
    if max_dimension < 1:
        raise ExactArithmeticError('max_dimension must be positive.')
    coefficients: dict[str, Any] = {}
    for n in range(1, max_dimension + 1):
        raw = wilson_character_coefficient_bounds(n, beta, series_terms)
        normalized = normalized_wilson_coefficient_bounds(n, beta, series_terms, exp_terms)
        coefficients[str(n)] = {
            'j2': n - 1, 'dimension': n,
            'raw_a_n': raw.to_payload(decimal_places),
            'normalized_exp_minus_beta_a_n': normalized.to_payload(decimal_places),
        }
    return {
        'rigor': 'RIGOROUS_RATIONAL_POSITIVE_SERIES',
        'formula': 'a_n(beta)=2*n*I_n(beta)/beta',
        'normalization': 'w_bar=exp(-beta)*sum_n a_n chi_n',
        'beta': {'numerator': beta.numerator, 'denominator': beta.denominator},
        'series_terms': series_terms, 'exp_terms': exp_terms,
        'coefficients': coefficients,
    }


def value_tail_enclosure(
    beta: Fraction, cutoff_n: int, series_terms: int = 96, exp_terms: int = 120,
) -> RationalInterval:
    if not isinstance(cutoff_n, int) or isinstance(cutoff_n, bool) or cutoff_n < 0:
        raise ExactArithmeticError('cutoff_n must be a nonnegative integer.')
    exp_beta = exp_positive_series_bounds(beta, exp_terms)
    partial_lo = Fraction(0)
    partial_hi = Fraction(0)
    for n in range(1, cutoff_n + 1):
        coefficient = wilson_character_coefficient_bounds(n, beta, series_terms)
        partial_lo += n * coefficient.lo
        partial_hi += n * coefficient.hi
    lower = max(Fraction(0), Fraction(1) - partial_hi / exp_beta.lo)
    upper = Fraction(1) - partial_lo / exp_beta.hi
    if upper < 0:
        raise ExactArithmeticError('Value-tail upper bound became negative; precision is inconsistent.')
    result = RationalInterval(lower, upper)
    result.assert_nonnegative()
    return result


def gradient_tail_enclosure(
    beta: Fraction, cutoff_n: int, exp_terms: int = 120,
) -> RationalInterval:
    if not isinstance(beta, Fraction) or beta <= 0:
        raise ExactArithmeticError('beta must be a positive Fraction.')
    if not isinstance(cutoff_n, int) or isinstance(cutoff_n, bool) or cutoff_n < 0:
        raise ExactArithmeticError('cutoff_n must be a nonnegative integer.')
    first_n = cutoff_n + 1
    half_beta = beta / 2
    first = Fraction(first_n**3, 1) * half_beta**first_n / factorial(first_n)
    ratio = half_beta * Fraction((first_n + 1) ** 2, first_n**3)
    if ratio >= 1:
        raise ExactArithmeticError('Gradient-tail n-series ratio is not contractive at this cutoff.')
    n_series_upper = first / (1 - ratio)
    correction = exp_positive_series_bounds(beta * beta / Fraction(4 * (cutoff_n + 2)), exp_terms)
    exp_beta = exp_positive_series_bounds(beta, exp_terms)
    upper = correction.hi * n_series_upper / (beta * exp_beta.lo)
    result = RationalInterval(Fraction(0), upper)
    result.assert_nonnegative()
    return result


def tail_table(
    beta: Fraction, cutoffs: tuple[int, ...], series_terms: int, exp_terms: int,
    decimal_places: int, kind: str,
) -> dict[str, Any]:
    if kind not in {'value', 'gradient'}:
        raise ExactArithmeticError(f'Unknown tail kind: {kind!r}')
    entries: dict[str, Any] = {}
    previous: RationalInterval | None = None
    for cutoff in cutoffs:
        interval = (
            value_tail_enclosure(beta, cutoff, series_terms, exp_terms)
            if kind == 'value'
            else gradient_tail_enclosure(beta, cutoff, exp_terms)
        )
        if previous is not None and interval.hi > previous.hi:
            raise ExactArithmeticError(f'{kind} tail upper bound is not monotone at cutoff {cutoff}.')
        previous = interval
        entry: dict[str, Any] = {'tail': interval.to_payload(decimal_places)}
        if kind == 'value':
            entry['cell_216'] = interval.scale(Fraction(216)).to_payload(decimal_places)
        else:
            entry['fine_link_6'] = interval.scale(Fraction(6)).to_payload(decimal_places)
            entry['coarse_cell_216'] = interval.scale(Fraction(216)).to_payload(decimal_places)
        entries[str(cutoff)] = entry
    return {
        'rigor': 'RIGOROUS_RATIONAL_ANALYTIC_BOUND',
        'kind': kind, 'proof': VALUE_TAIL_PROOF if kind == 'value' else GRADIENT_TAIL_PROOF,
        'metric_normalization': 'C2(j)=j(j+1)',
        'entries': entries,
    }


In [ ]:
%%writefile src/exact_2d_rg.py
from __future__ import annotations

from fractions import Fraction
from typing import Any, Iterable

from .exact_arithmetic import ExactArithmeticError, RationalInterval
from .su2_special_functions import wilson_character_coefficient_bounds


def normalized_fourier_ratio(
    beta: Fraction, dimension_n: int, series_terms: int = 96,
) -> RationalInterval:
    if not isinstance(dimension_n, int) or isinstance(dimension_n, bool) or dimension_n < 2:
        raise ExactArithmeticError('Nontrivial irrep dimension n must be at least two.')
    trivial = wilson_character_coefficient_bounds(1, beta, series_terms)
    coefficient = wilson_character_coefficient_bounds(dimension_n, beta, series_terms)
    denominator = trivial.scale(Fraction(dimension_n))
    ratio = coefficient.divide_positive(denominator)
    if ratio.lo < 0 or ratio.hi > 1:
        raise ExactArithmeticError('Normalized Fourier ratio left the expected [0,1] range.')
    return ratio


def exact_2d_rg_trajectory(
    beta: Fraction, dimensions: Iterable[int], steps: int, series_terms: int = 96,
) -> dict[int, tuple[RationalInterval, ...]]:
    if not isinstance(steps, int) or isinstance(steps, bool) or steps < 0:
        raise ExactArithmeticError('RG steps must be a nonnegative integer.')
    result: dict[int, tuple[RationalInterval, ...]] = {}
    for dimension in dimensions:
        current = normalized_fourier_ratio(beta, dimension, series_terms)
        trajectory = [current]
        for _ in range(steps):
            current = current.positive_power(4)
            trajectory.append(current)
        result[dimension] = tuple(trajectory)
    return result


def trajectory_payload(
    beta: Fraction, dimensions: tuple[int, ...], steps: int, series_terms: int, decimal_places: int,
) -> dict[str, Any]:
    trajectories = exact_2d_rg_trajectory(beta, dimensions, steps, series_terms)
    return {
        'rigor': 'RIGOROUS_RATIONAL_INTERVAL_RECURRENCE',
        'theorem': (
            'For normalized central Fourier ratios r_n=a_n/(n*a_1), '
            'fourfold Haar convolution for a 2x2 block gives r_n_next=r_n^4.'
        ),
        'beta': {'numerator': beta.numerator, 'denominator': beta.denominator},
        'steps': steps,
        'trajectories': {
            str(n): [interval.to_payload(decimal_places) for interval in trajectory]
            for n, trajectory in trajectories.items()
        },
    }


In [ ]:
%%writefile src/m1_verifier.py
from __future__ import annotations

import importlib.util
from dataclasses import dataclass
from fractions import Fraction
from math import factorial
from typing import Any


class IndependentVerificationError(RuntimeError):
    '''Raised when the independent M1 convolution path does not enclose the primary result.'''


@dataclass(frozen=True, slots=True)
class VerifierInterval:
    lo: Fraction
    hi: Fraction

    def __post_init__(self) -> None:
        if not isinstance(self.lo, Fraction) or not isinstance(self.hi, Fraction) or self.lo > self.hi:
            raise IndependentVerificationError('Invalid independent-verifier interval.')

    def multiply(self, other: 'VerifierInterval') -> 'VerifierInterval':
        products = (self.lo * other.lo, self.lo * other.hi, self.hi * other.lo, self.hi * other.hi)
        return VerifierInterval(min(products), max(products))

    def fourth_power(self) -> 'VerifierInterval':
        if self.lo < 0:
            raise IndependentVerificationError('Verifier fourth power requires nonnegative input.')
        square = self.multiply(self)
        return square.multiply(square)

    def divide_positive(self, other: 'VerifierInterval') -> 'VerifierInterval':
        if other.lo <= 0:
            raise IndependentVerificationError('Verifier denominator is not strictly positive.')
        return self.multiply(VerifierInterval(Fraction(1, other.hi), Fraction(1, other.lo)))

    def scale_positive(self, value: Fraction) -> 'VerifierInterval':
        if value < 0:
            raise IndependentVerificationError('Verifier positive scaling received a negative value.')
        return VerifierInterval(self.lo * value, self.hi * value)


def _direct_bessel_interval(n: int, beta: Fraction, terms: int) -> VerifierInterval:
    half = beta / 2
    total = Fraction(0)
    for k in range(terms + 1):
        total += half ** (2 * k + n) / (factorial(k) * factorial(n + k))
    next_term = half ** (2 * (terms + 1) + n) / (factorial(terms + 1) * factorial(n + terms + 1))
    next_ratio = half * half / ((terms + 2) * (n + terms + 2))
    if next_ratio >= 1:
        raise IndependentVerificationError('Independent Bessel remainder is not contractive.')
    return VerifierInterval(total, total + next_term / (1 - next_ratio))


def _direct_coefficient_interval(n: int, beta: Fraction, terms: int) -> VerifierInterval:
    bessel = _direct_bessel_interval(n, beta, terms)
    return bessel.scale_positive(Fraction(2 * n, 1) / beta)


@dataclass(frozen=True, slots=True)
class DiagonalConvolutionOperator:
    coefficients: dict[int, VerifierInterval]

    def fourfold_block(self) -> 'DiagonalConvolutionOperator':
        blocked: dict[int, VerifierInterval] = {}
        for n, coefficient in self.coefficients.items():
            blocked[n] = coefficient.fourth_power().scale_positive(Fraction(1, n**3))
        return DiagonalConvolutionOperator(blocked)

    def normalized_ratio(self, n: int) -> VerifierInterval:
        if 1 not in self.coefficients or n not in self.coefficients:
            raise IndependentVerificationError('Requested convolution sector is absent.')
        denominator = self.coefficients[1].scale_positive(Fraction(n))
        return self.coefficients[n].divide_positive(denominator)


def _fraction_from_hex(payload: dict[str, Any]) -> Fraction:
    return Fraction(int(payload['numerator_hex'], 16), int(payload['denominator_hex'], 16))


def _primary_interval(payload: dict[str, Any]) -> VerifierInterval:
    return VerifierInterval(_fraction_from_hex(payload['lo']), _fraction_from_hex(payload['hi']))


def _verifier_interval_payload(interval: VerifierInterval) -> dict[str, Any]:
    return {
        'lo': {
            'numerator_hex': format(interval.lo.numerator, 'x'),
            'denominator_hex': format(interval.lo.denominator, 'x'),
        },
        'hi': {
            'numerator_hex': format(interval.hi.numerator, 'x'),
            'denominator_hex': format(interval.hi.denominator, 'x'),
        },
    }


def independent_convolution_verify(
    primary_payload: dict[str, Any], beta: Fraction, dimensions: tuple[int, ...],
    steps: int, verifier_terms: int,
) -> dict[str, Any]:
    coefficients = {
        n: _direct_coefficient_interval(n, beta, verifier_terms)
        for n in (1,) + dimensions
    }
    operator = DiagonalConvolutionOperator(coefficients)
    verifier_trajectories: dict[int, list[VerifierInterval]] = {
        n: [operator.normalized_ratio(n)] for n in dimensions
    }
    for _ in range(steps):
        operator = operator.fourfold_block()
        for n in dimensions:
            verifier_trajectories[n].append(operator.normalized_ratio(n))
    checks: list[dict[str, Any]] = []
    for n in dimensions:
        primary_steps = primary_payload['trajectories'][str(n)]
        if len(primary_steps) != steps + 1:
            raise IndependentVerificationError('Primary trajectory length mismatch.')
        for step, primary_step in enumerate(primary_steps):
            primary = _primary_interval(primary_step)
            independent = verifier_trajectories[n][step]
            overlaps = max(primary.lo, independent.lo) <= min(primary.hi, independent.hi)
            primary_inside = independent.lo <= primary.lo and primary.hi <= independent.hi
            if not overlaps or not primary_inside:
                raise IndependentVerificationError(
                    f'Independent convolution failed containment for n={n}, step={step}.'
                )
            checks.append({
                'dimension': n, 'step': step, 'overlaps': overlaps,
                'primary_inside_independent': primary_inside,
            })
    return {
        'status': 'PASS',
        'method': (
            'Independent direct-factorial Bessel sums plus a finite diagonal Peter-Weyl '
            'convolution operator using chi_n*chi_m=delta_nm*chi_n/n.'
        ),
        'does_not_call_primary_recurrence': True,
        'verifier_series_terms': verifier_terms,
        'arb_status': 'AVAILABLE_NOT_USED' if importlib.util.find_spec('flint') else 'NOT_AVAILABLE_NOT_REQUIRED',
        'independent_trajectories': {
            str(n): [_verifier_interval_payload(interval) for interval in trajectory]
            for n, trajectory in verifier_trajectories.items()
        },
        'checks': checks,
    }


In [ ]:
%%writefile src/m1_reporting.py
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

from .checkpoint import CheckpointSaveResult, RunState
from .common import atomic_write_json, atomic_write_text, read_json, sha256_file, utc_now
from .m1_config import M1Config
from .reporting import peak_memory_report
from .work_queue import WorkQueue

REQUIRED_PHASES = (
    'M1_COEFFICIENT_BATCH', 'M1_VALUE_TAIL', 'M1_GRADIENT_TAIL',
    'M1_RG_TRAJECTORY', 'M1_INDEPENDENT_VERIFY', 'M1_REPORT',
)


def load_phase_results(run_root: Path, queue: WorkQueue) -> dict[str, dict[str, Any]]:
    results: dict[str, dict[str, Any]] = {}
    for item in queue.items.values():
        if item.status != 'done' or not item.result_relpath:
            continue
        if item.phase in results:
            raise RuntimeError(f'Duplicate completed M1 phase artifact: {item.phase}')
        result_path = (run_root / item.result_relpath).resolve()
        try:
            result_path.relative_to(run_root.resolve())
        except ValueError as exc:
            raise RuntimeError(f'M1 phase artifact escapes run root: {item.phase}') from exc
        if not result_path.is_file() or sha256_file(result_path) != item.result_sha256:
            raise RuntimeError(f'M1 phase artifact hash mismatch: {item.phase}')
        payload = read_json(result_path)
        if not isinstance(payload, dict) or payload.get('phase') != item.phase:
            raise RuntimeError(f'Malformed M1 phase artifact: {item.phase}')
        results[item.phase] = payload
    return results


def _acceptance_gates(
    state: RunState, queue: WorkQueue, results: dict[str, dict[str, Any]], test_report: dict[str, Any],
) -> dict[str, bool]:
    statuses = [item.status for item in queue.items.values()]
    return {
        'm0_regression_cpu_tests': test_report.get('m0_regression_cpu_suite') == 'PASS',
        'required_cpu_tests': test_report.get('m1_required_cpu_suite') == 'PASS',
        'gpu_smoke_if_available': test_report.get('optional_gpu_suite') in {'PASS', 'NOT_RUN_NO_CUDA'},
        'fresh_process_resume': test_report.get('m1_fresh_process_resume') == 'PASS',
        'coefficient_enclosures_rigorous': results.get('M1_COEFFICIENT_BATCH', {}).get('result', {}).get('rigor') == 'RIGOROUS_RATIONAL_POSITIVE_SERIES',
        'value_tail_rigorous': results.get('M1_VALUE_TAIL', {}).get('result', {}).get('rigor') == 'RIGOROUS_RATIONAL_ANALYTIC_BOUND',
        'gradient_tail_rigorous': results.get('M1_GRADIENT_TAIL', {}).get('result', {}).get('rigor') == 'RIGOROUS_RATIONAL_ANALYTIC_BOUND',
        'exact_2d_rg_rigorous': results.get('M1_RG_TRAJECTORY', {}).get('result', {}).get('rigor') == 'RIGOROUS_RATIONAL_INTERVAL_RECURRENCE',
        'independent_verifier': results.get('M1_INDEPENDENT_VERIFY', {}).get('result', {}).get('status') == 'PASS',
        'report_work_item_ready': results.get('M1_REPORT', {}).get('result', {}).get('status') == 'READY',
        'queue_complete': bool(statuses) and all(status == 'done' for status in statuses),
        'not_certified_invariant': state.certification_status == 'NOT_CERTIFIED',
    }


def validate_m1_acceptance(
    state: RunState, queue: WorkQueue, results: dict[str, dict[str, Any]], test_report: dict[str, Any],
) -> dict[str, bool]:
    missing = [phase for phase in REQUIRED_PHASES if phase not in results]
    if missing:
        raise RuntimeError(f'M1 acceptance is missing phase artifacts: {missing}')
    gates = _acceptance_gates(state, queue, results, test_report)
    failed = [name for name, passed in gates.items() if not passed]
    if failed:
        raise RuntimeError(f'M1 acceptance gates failed closed: {failed}')
    return gates


def _markdown_report(report: dict[str, Any]) -> str:
    results = report['results']
    value_entries = results['M1_VALUE_TAIL']['result']['entries']
    gradient_entries = results['M1_GRADIENT_TAIL']['result']['entries']
    trajectories = results['M1_RG_TRAJECTORY']['result']['trajectories']
    lines = [
        '# M1 exact 2D SU(2) benchmark report', '',
        f"- run ID: `{report['run_id']}`",
        f"- parent M0 run: `{report['parent']['parent_run_id']}`",
        f"- parent checkpoint: `{report['parent']['parent_checkpoint']}`",
        '- status: `NOT_CERTIFIED`',
        '- scope: M1 only; no 4D armillary/RG/mass-gap claim', '',
        '## Conventions', '',
        r'\(j=j2/2\), \(d_j=j2+1\), \(C_2=j(j+1)\), \(\mathrm{Tr}U=2\cos\theta\).', '',
        r'\(\bar w_\beta=e^{\beta(\cos\theta-1)}\), '        r'\(a_n=2n I_n(\beta)/\beta\), and \(r_n=a_n/(n a_1)\).', '',
        'All proof-path endpoints are exact rational numbers serialized in hexadecimal. Decimal strings are outward-rounded displays.', '',
        '## Proof formulas used by the implementation', '',
        r'Positive series: \(I_n(\beta)=\sum_{k\ge0}(\beta/2)^{2k+n}/(k!(n+k)!)\). A decreasing next-term ratio bounds the omitted positive tail geometrically.', '',
        r'Value tail: \(\|\bar w-P_N\bar w\|_\infty\le e^{-\beta}\sum_{n>N}2n^2 I_n(\beta)/\beta\).', '',
        r'Gradient tail: use the explicit weight-sum bound \(\|\nabla\chi_j\|_\infty\le n^2/2\) and \(I_n\le(\beta/2)^n e^{\beta^2/(4(n+1))}/n!\).', '',
        r'Independent convolution: \(\chi_n*\chi_m=\delta_{nm}\chi_n/n\), so fourfold blocking sends the coefficient to \(a_n^4/n^3\) and the normalized ratio to \(r_n^4\).', '',
        '## Value tail', '', '| cutoff n | tail lower | tail upper |', '|---:|---:|---:|',
    ]
    for cutoff, entry in value_entries.items():
        interval = entry['tail']
        lines.append(f"| {cutoff} | {interval['decimal_lo']} | {interval['decimal_hi']} |")
    lines.extend(['', '## Casimir-gradient tail', '', '| cutoff n | tail lower | tail upper | 6× upper | 216× upper |', '|---:|---:|---:|---:|---:|'])
    for cutoff, entry in gradient_entries.items():
        interval = entry['tail']; fine = entry['fine_link_6']; cell = entry['coarse_cell_216']
        lines.append(f"| {cutoff} | {interval['decimal_lo']} | {interval['decimal_hi']} | {fine['decimal_hi']} | {cell['decimal_hi']} |")
    lines.extend(['', 'The factor 6 is the fine-link contact count. The 216 factor is recorded only as a deliberately coarse cell-wide comparison and is not substituted for the contact count.', '', '## Exact 2D 2×2 RG', ''])
    for dimension, steps in trajectories.items():
        lines.extend([f'### dimension n={dimension}', '', '| step | lower | upper |', '|---:|---:|---:|'])
        for step, interval in enumerate(steps):
            lines.append(f"| {step} | {interval['decimal_lo']} | {interval['decimal_hi']} |")
        lines.append('')
    verifier = results['M1_INDEPENDENT_VERIFY']['result']
    lines.extend([
        '## Independent verification', '',
        f"- status: `{verifier['status']}`",
        f"- method: {verifier['method']}",
        f"- Arb: `{verifier['arb_status']}`", '',
        '## Proven statements', '',
        '- Wilson character coefficients are enclosed by positive rational Bessel series with a term-ratio remainder.',
        '- For the listed cutoffs, the representation value tail and Casimir-gradient tail have rigorous rational upper bounds.',
        '- For n=2,3,4 and steps 0–3, the exact 2D recurrence is enclosed and independently reproduced by diagonal convolution.', '',
        '## Limitations', '',
        '- No four-dimensional armillary tensor or four-dimensional RG step is constructed.',
        '- No influence matrix, mass-gap bound, thermodynamic limit, or continuum statement is proved.',
        '- `CERTIFIED` remains forbidden; the milestone status is `NOT_CERTIFIED`.', '',
    ])
    return '\n'.join(lines)


def write_m1_report_package(
    run_root: Path, config: M1Config, state: RunState, queue: WorkQueue,
    test_report: dict[str, Any], last_checkpoint: CheckpointSaveResult, manifest: dict[str, Any],
) -> dict[str, str]:
    results = load_phase_results(run_root, queue)
    gates = validate_m1_acceptance(state, queue, results, test_report)
    report = {
        'schema_version': 1, 'milestone': 'M1', 'phase': state.phase,
        'run_id': state.run_id, 'generated_at': utc_now(),
        'certification_status': 'NOT_CERTIFIED',
        'parent': {key: manifest[key] for key in (
            'parent_milestone', 'parent_run_id', 'parent_checkpoint', 'parent_checkpoint_path',
            'parent_checkpoint_hash_manifest_sha256',
        )},
        'config': config.canonical_payload(), 'config_hash': config.config_hash(),
        'source_hash': manifest['source_hash'],
        'governing_document_hashes': manifest['governing_document_hashes'],
        'reference_artifact_hashes': manifest['reference_artifact_hashes'],
        'm0_acceptance_record_sha256': manifest['m0_acceptance_record_sha256'],
        'results': results, 'tests': test_report, 'acceptance_gates': gates,
        'proof_artifact_hashes': {
            item.phase: item.result_sha256 for item in queue.items.values()
            if item.status == 'done' and item.result_sha256 is not None
        },
        'checkpoint': {
            'path': str(last_checkpoint.path), 'index': last_checkpoint.index,
            'size_bytes': last_checkpoint.size_bytes, 'save_s': last_checkpoint.save_s,
            'verify_s': last_checkpoint.verify_s,
        },
        'memory': peak_memory_report(),
        'rigorous_results': [
            'positive-series Wilson coefficient enclosures', 'value-tail enclosures',
            'Casimir-gradient tail upper bounds', 'exact 2D r_n -> r_n^4 trajectories',
            'independent finite diagonal-convolution containment',
        ],
        'heuristic_results': [],
        'unresolved_issues': [
            'P0 transitive checkpoint hash-chain certification is not claimed at M1.',
            'Arb is optional; two independent rational paths are the required M1 verifier.',
            'No four-dimensional RG enclosure has been constructed.',
        ],
        'remaining_todos': ['M2 low-cutoff armillary', 'M3 GPU Triad-ATRG', 'M4 forward AD', 'M5 one-step validation', 'M6 multi-step certificate'],
    }
    report_dir = run_root / 'reports'; report_dir.mkdir(parents=True, exist_ok=True)
    json_path = report_dir / 'M1_report.json'; md_path = report_dir / 'M1_report.md'
    acceptance_path = report_dir / 'M1_acceptance.json'
    atomic_write_json(json_path, report)
    atomic_write_text(md_path, _markdown_report(report))
    atomic_write_json(acceptance_path, {
        'milestone': 'M1', 'phase': 'M1_COMPLETE', 'status': 'PASS',
        'certification_status': 'NOT_CERTIFIED', 'gates': gates, 'generated_at': utc_now(),
    })
    return {'json': str(json_path), 'markdown': str(md_path), 'acceptance': str(acceptance_path)}


def write_m1_session_artifacts(
    run_root: Path, state: RunState, queue: WorkQueue, stop_reason: str,
    elapsed_s: float, remaining_s: float, persistent_root: Path, project_root: Path,
) -> dict[str, str]:
    report_dir = run_root / 'reports'; report_dir.mkdir(parents=True, exist_ok=True)
    counts = {status: sum(item.status == status for item in queue.items.values()) for status in ('pending', 'running', 'done', 'failed', 'blocked_resource')}
    unfinished = [item.item_id for item in queue.items.values() if item.status != 'done']
    summary_path = report_dir / 'session_summary.json'; metrics_path = report_dir / 'latest_metrics.json'
    plan_path = report_dir / 'next_session_plan.md'
    atomic_write_json(summary_path, {
        'milestone': 'M1', 'run_id': state.run_id, 'phase': state.phase,
        'certification_status': 'NOT_CERTIFIED', 'stop_reason': stop_reason,
        'elapsed_s': elapsed_s, 'remaining_s': remaining_s, 'queue': counts,
        'unfinished_work_items': unfinished, 'generated_at': utc_now(),
    })
    atomic_write_json(metrics_path, {
        'milestone': 'M1', 'run_id': state.run_id, 'memory': peak_memory_report(),
        'rigorous_bounds': sorted(state.bounds), 'approximate_spectral_radius': None,
        'certification_status': 'NOT_CERTIFIED', 'generated_at': utc_now(),
    })
    if state.phase == 'M1_COMPLETE':
        plan = '# Next session plan\n\nM1 is complete. Review M1_acceptance.json and do not start M2 automatically.\n'
    else:
        plan = (
            '# Next session plan\n\n'
            f'1. Reopen `{project_root}` with the same Paperspace runtime.\n'
            f'2. Set `VALIDATED_RG_PERSIST_ROOT={persistent_root}`.\n'
            '3. Set `VALIDATED_RG_PERSIST_ACK=I_CONFIRM_THIS_PATH_IS_PERSISTENT`.\n'
            f'4. Set `VALIDATED_RG_M1_RUN_ID={state.run_id}`.\n'
            '5. Use a fresh kernel, run the notebook top-to-bottom, then call `m1_orchestrator.run_until_checkpoint()`.\n'
        )
    atomic_write_text(plan_path, plan)
    return {'session_summary': str(summary_path), 'latest_metrics': str(metrics_path), 'next_session_plan': str(plan_path)}


In [ ]:
%%writefile src/m1_orchestrator.py
from __future__ import annotations

import json
import os
import random
import shutil
import time
import uuid
from datetime import datetime, timezone
from fractions import Fraction
from pathlib import Path
from typing import Any

import numpy as np

from .checkpoint import CheckpointManager, CheckpointSaveResult, RunState
from .common import (
    atomic_write_json, canonical_json_bytes, fsync_directory, hash_tree, read_json,
    safe_component, sha256_bytes, sha256_file, utc_now,
)
from .exact_2d_rg import trajectory_payload
from .m1_config import M1Config
from .m1_reporting import (
    load_phase_results, validate_m1_acceptance, write_m1_report_package,
    write_m1_session_artifacts,
)
from .m1_verifier import independent_convolution_verify
from .orchestrator import governing_document_hashes, reference_artifact_hashes
from .reporting import JsonlLogger
from .runtime import environment_info, runtime_compatibility_signature
from .session_guard import SessionGuard, SessionState
from .su2_representations import CONVENTION
from .tail_bounds import coefficient_enclosures, tail_table
from .work_queue import WorkItem, WorkQueue

try:
    import torch
except ImportError:
    torch = None

M1_PHASE_ORDER = (
    'M1_COEFFICIENT_BATCH', 'M1_VALUE_TAIL', 'M1_GRADIENT_TAIL',
    'M1_RG_TRAJECTORY', 'M1_INDEPENDENT_VERIFY', 'M1_REPORT',
)
M0_ACCEPTANCE_RECORD = 'audit/m0_accepted_parent.json'
M1_NOTEBOOK_HASH_POLICY = 'canonical_nbformat4_cell_type_source_tags_v1'


class M1CompatibilityError(RuntimeError):
    '''Raised when an M1 run or its accepted M0 parent is incompatible.'''


def _seed_everything(seed: int) -> None:
    random.seed(seed); np.random.seed(seed)
    if torch is not None:
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
            torch.cuda.reset_peak_memory_stats()


def _notebook_hash(project_root: Path) -> str | None:
    path = project_root / 'notebooks/20_m1_exact_2d.ipynb'
    if not path.is_file():
        return None
    payload = read_json(path)
    cells = payload.get('cells') if isinstance(payload, dict) else None
    if not isinstance(payload, dict) or payload.get('nbformat') != 4 or not isinstance(cells, list):
        raise M1CompatibilityError('M1 notebook must be a valid nbformat 4 document.')
    identity_cells: list[dict[str, Any]] = []
    for index, cell in enumerate(cells):
        if not isinstance(cell, dict):
            raise M1CompatibilityError(f'M1 notebook cell {index} is not a mapping.')
        cell_type = cell.get('cell_type')
        source = cell.get('source')
        metadata = cell.get('metadata', {})
        if cell_type not in {'code', 'markdown', 'raw'}:
            raise M1CompatibilityError(f'M1 notebook cell {index} has an invalid type.')
        if isinstance(source, str):
            normalized_source = source
        elif isinstance(source, list) and all(isinstance(line, str) for line in source):
            normalized_source = ''.join(source)
        else:
            raise M1CompatibilityError(f'M1 notebook cell {index} has invalid source text.')
        if not isinstance(metadata, dict):
            raise M1CompatibilityError(f'M1 notebook cell {index} has invalid metadata.')
        tags = metadata.get('tags', [])
        if not isinstance(tags, list) or any(not isinstance(tag, str) for tag in tags):
            raise M1CompatibilityError(f'M1 notebook cell {index} has invalid execution tags.')
        identity_cells.append({
            'cell_type': cell_type, 'source': normalized_source, 'tags': tags,
        })
    identity = {
        'policy': M1_NOTEBOOK_HASH_POLICY, 'nbformat': 4, 'cells': identity_cells,
    }
    return sha256_bytes(canonical_json_bytes(identity))


def _validate_acceptance_record(project_root: Path, config: M1Config) -> str:
    path = project_root / M0_ACCEPTANCE_RECORD
    if not path.is_file():
        raise M1CompatibilityError('M0 acceptance record is missing.')
    payload = read_json(path)
    expected = {
        'parent_milestone': config.parent_milestone,
        'parent_run_id': config.parent_run_id,
        'parent_checkpoint': config.parent_checkpoint,
        'accepted_phase': 'M0_COMPLETE',
        'certification_status': 'NOT_CERTIFIED',
        'restart_test_status': 'PASS',
    }
    if not isinstance(payload, dict) or any(payload.get(key) != value for key, value in expected.items()):
        raise M1CompatibilityError('M0 acceptance record identity/status is invalid.')
    if payload.get('independent_checkpoint_audit_performed') is not False:
        raise M1CompatibilityError('M0 acceptance audit scope was silently changed.')
    return sha256_file(path)


def verify_accepted_m0_checkpoint(config: M1Config) -> str:
    checkpoint = Path(config.parent_checkpoint_path).expanduser().resolve()
    if checkpoint.name != config.parent_checkpoint or checkpoint.is_symlink() or not checkpoint.is_dir():
        raise M1CompatibilityError(f'Accepted M0 checkpoint path is unavailable or unsafe: {checkpoint}')
    if not (checkpoint / 'COMMITTED').is_file() or not (checkpoint / 'hashes.json').is_file():
        raise M1CompatibilityError('Accepted M0 checkpoint is not committed or lacks hashes.json.')
    if any(path.is_symlink() for path in checkpoint.rglob('*')):
        raise M1CompatibilityError('Accepted M0 checkpoint contains a symlink.')
    expected = read_json(checkpoint / 'hashes.json')
    if not isinstance(expected, dict) or any(
        not isinstance(name, str) or not isinstance(digest, str) or len(digest) != 64
        for name, digest in expected.items()
    ):
        raise M1CompatibilityError('Accepted M0 checkpoint hash manifest is malformed.')
    actual_files = {
        path.relative_to(checkpoint).as_posix() for path in checkpoint.rglob('*')
        if path.is_file() and path.name not in {'hashes.json', 'COMMITTED'}
    }
    if actual_files != set(expected):
        raise M1CompatibilityError('Accepted M0 checkpoint file set differs from hashes.json.')
    for relative, digest in expected.items():
        if sha256_file(checkpoint / relative) != digest:
            raise M1CompatibilityError(f'Accepted M0 checkpoint hash mismatch: {relative}')
    state = read_json(checkpoint / 'state.json')
    if not isinstance(state, dict) or (
        state.get('run_id') != config.parent_run_id
        or state.get('phase') != 'M0_COMPLETE'
        or state.get('certification_status') != 'NOT_CERTIFIED'
        or state.get('checkpoint_index') != 14
    ):
        raise M1CompatibilityError('Accepted M0 checkpoint state does not match the acceptance record.')
    queue = WorkQueue.from_payload(read_json(checkpoint / 'work_queue.json'))
    counts = {status: sum(item.status == status for item in queue.items.values()) for status in ('done', 'pending', 'running', 'failed')}
    if counts != {'done': 6, 'pending': 0, 'running': 0, 'failed': 0}:
        raise M1CompatibilityError(f'Accepted M0 queue is not complete: {counts}')
    parent_run_root = checkpoint.parents[1]
    for item in queue.items.values():
        if not item.result_relpath or not item.result_sha256:
            raise M1CompatibilityError(f'Accepted M0 done item lacks result metadata: {item.item_id}')
        result_path = (parent_run_root / item.result_relpath).resolve()
        try:
            result_path.relative_to(parent_run_root.resolve())
        except ValueError as exc:
            raise M1CompatibilityError('Accepted M0 result path escapes its run root.') from exc
        marker_path = parent_run_root / 'work_items' / f'{item.item_id}.done'
        if not result_path.is_file() or sha256_file(result_path) != item.result_sha256:
            raise M1CompatibilityError(f'Accepted M0 result artifact is missing or corrupt: {item.item_id}')
        marker = read_json(marker_path) if marker_path.is_file() else None
        if not isinstance(marker, dict) or (
            marker.get('item_id') != item.item_id
            or marker.get('result_relpath') != item.result_relpath
            or marker.get('result_sha256') != item.result_sha256
        ):
            raise M1CompatibilityError(f'Accepted M0 done marker is missing or inconsistent: {item.item_id}')
    return sha256_file(checkpoint / 'hashes.json')


class M1Orchestrator:
    def __init__(
        self, persistent_root: Path, run_root: Path, project_root: Path, config: M1Config,
        state: RunState, queue: WorkQueue, checkpoints: CheckpointManager,
        test_report: dict[str, Any], manifest: dict[str, Any],
    ) -> None:
        self.persistent_root = persistent_root
        self.run_root = run_root
        self.project_root = project_root
        self.config = config
        self.state = state
        self.queue = queue
        self.checkpoints = checkpoints
        self.test_report = test_report
        self.manifest = manifest
        self.guard = SessionGuard(config)
        self.logger = JsonlLogger(run_root / 'logs' / 'events.jsonl')
        self.last_checkpoint: CheckpointSaveResult | None = None

    def checkpoint(self, reason: str) -> CheckpointSaveResult:
        self.state.assert_m1_safe()
        self.state.notes.append(f'{utc_now()} checkpoint: {reason}')
        result = self.checkpoints.save(self.state, self.queue, {})
        self.guard.mark_checkpoint(); self.last_checkpoint = result
        self.logger.emit('m1_checkpoint_committed', run_id=self.state.run_id, checkpoint=result.index, reason=reason)
        print(f'M1 checkpoint {result.index:06d} committed and verified: {result.path}')
        return result

    def _next_pending(self) -> WorkItem | None:
        for phase in M1_PHASE_ORDER:
            candidates = sorted(
                (item for item in self.queue.items.values() if item.phase == phase and item.status == 'pending'),
                key=lambda item: item.item_id,
            )
            if candidates:
                return candidates[0]
        return None

    def _phase_result(self, phase: str) -> dict[str, Any]:
        results = load_phase_results(self.run_root, self.queue)
        if phase not in results:
            raise RuntimeError(f'Required prior M1 phase is incomplete: {phase}')
        return results[phase]['result']

    def _compute_phase(self, item: WorkItem) -> dict[str, Any]:
        beta = Fraction(self.config.beta_numerator, self.config.beta_denominator)
        if item.phase == 'M1_COEFFICIENT_BATCH':
            return coefficient_enclosures(
                beta, max(self.config.cutoffs), self.config.coefficient_series_terms,
                self.config.exp_series_terms, self.config.decimal_places,
            )
        if item.phase == 'M1_VALUE_TAIL':
            return tail_table(
                beta, self.config.cutoffs, self.config.coefficient_series_terms,
                self.config.exp_series_terms, self.config.decimal_places, 'value',
            )
        if item.phase == 'M1_GRADIENT_TAIL':
            return tail_table(
                beta, self.config.cutoffs, self.config.coefficient_series_terms,
                self.config.exp_series_terms, self.config.decimal_places, 'gradient',
            )
        if item.phase == 'M1_RG_TRAJECTORY':
            return trajectory_payload(
                beta, self.config.dimensions, self.config.rg_steps,
                self.config.coefficient_series_terms, self.config.decimal_places,
            )
        if item.phase == 'M1_INDEPENDENT_VERIFY':
            return independent_convolution_verify(
                self._phase_result('M1_RG_TRAJECTORY'), beta, self.config.dimensions,
                self.config.rg_steps, self.config.verifier_series_terms,
            )
        if item.phase == 'M1_REPORT':
            required = M1_PHASE_ORDER[:-1]
            results = load_phase_results(self.run_root, self.queue)
            missing = [phase for phase in required if phase not in results]
            if missing:
                raise RuntimeError(f'M1 report work item is missing inputs: {missing}')
            return {'status': 'READY', 'input_phases': list(required)}
        raise RuntimeError(f'Unknown M1 work phase: {item.phase}')

    def _execute_item(self, item: WorkItem) -> tuple[str, str]:
        parent = self.run_root / 'artifacts' / item.item_id
        parent.mkdir(parents=True, exist_ok=True)
        temporary = parent / f'.tmp-attempt-{item.attempts:03d}-{uuid.uuid4().hex}'
        final = parent / f'attempt_{item.attempts:03d}'
        if final.exists():
            raise RuntimeError(f'M1 attempt output already exists: {final}')
        temporary.mkdir(parents=False, exist_ok=False)
        try:
            result_file = temporary / 'result.json'
            atomic_write_json(result_file, {
                'schema_version': 1, 'milestone': 'M1', 'phase': item.phase,
                'item_id': item.item_id, 'config_hash': self.config.config_hash(),
                'certification_status': 'NOT_CERTIFIED', 'generated_at': utc_now(),
                'result': self._compute_phase(item),
            })
            fsync_directory(temporary)
            os.replace(temporary, final); fsync_directory(parent)
            committed = final / 'result.json'
            relative = committed.relative_to(self.run_root).as_posix(); digest = sha256_file(committed)
            atomic_write_json(self.run_root / 'work_items' / f'{item.item_id}.done', {
                'item_id': item.item_id, 'result_relpath': relative,
                'result_sha256': digest, 'completed_at': utc_now(),
            })
            if sha256_file(committed) != digest:
                raise RuntimeError('M1 result verification failed after commit.')
            return relative, digest
        except Exception:
            if temporary.exists():
                shutil.rmtree(temporary, ignore_errors=True)
            raise

    def run_one_item_for_test(self) -> str | None:
        item = self._next_pending()
        if item is None:
            return None
        item.attempts += 1; item.status = 'running'
        self.checkpoint(f'before test item {item.phase}')
        relative, digest = self._execute_item(item)
        item.result_relpath = relative; item.result_sha256 = digest; item.status = 'done'
        self.checkpoint(f'after test item {item.phase}')
        return item.phase

    def _summary(self, stop_reason: str) -> dict[str, Any]:
        elapsed = self.guard.elapsed_s(); remaining = self.guard.remaining_s()
        artifacts = write_m1_session_artifacts(
            self.run_root, self.state, self.queue, stop_reason, elapsed, remaining,
            self.persistent_root, self.project_root,
        )
        summary = {
            'milestone': 'M1', 'run_id': self.state.run_id, 'phase': self.state.phase,
            'checkpoint_index': self.state.checkpoint_index,
            'certification_status': 'NOT_CERTIFIED', 'stop_reason': stop_reason,
            'elapsed_s': elapsed, 'remaining_s': remaining, 'session_artifacts': artifacts,
        }
        print(json.dumps(summary, ensure_ascii=False, indent=2, allow_nan=False))
        return summary

    def run_until_checkpoint(self) -> dict[str, Any]:
        self.state.assert_m1_safe()
        if self.state.phase == 'M1_COMPLETE' and (self.run_root / 'reports' / 'M1_acceptance.json').is_file():
            return self._summary('M1 already complete; no work or checkpoint was started')
        self.state.phase = 'M1_RUNNING'
        self.logger.emit('m1_session_started', run_id=self.state.run_id)
        while True:
            session_state = self.guard.state()
            if session_state is SessionState.RETURN:
                return self._summary('hard return threshold reached; using last committed checkpoint')
            if session_state in {SessionState.DRAIN, SessionState.FINAL_SAVE}:
                self.checkpoint(f'M1 session state {session_state.value}')
                return self._summary(f'{session_state.value.lower()} checkpoint complete')
            if self.guard.checkpoint_due():
                self.checkpoint('periodic 15-minute M1 checkpoint')
            item = self._next_pending()
            if item is None:
                incomplete = [queued for queued in self.queue.items.values() if queued.status != 'done']
                if incomplete:
                    self.checkpoint('M1 queue has no runnable item but is incomplete')
                    raise RuntimeError('M1 cannot complete with failed/blocked/running work items.')
                results = load_phase_results(self.run_root, self.queue)
                validate_m1_acceptance(self.state, self.queue, results, self.test_report)
                self.state.bounds = {
                    'coefficient_enclosures': 'RIGOROUS_RATIONAL_POSITIVE_SERIES',
                    'value_tail': 'RIGOROUS_RATIONAL_ANALYTIC_BOUND',
                    'gradient_tail': 'RIGOROUS_RATIONAL_ANALYTIC_BOUND',
                    'exact_2d_rg': 'RIGOROUS_RATIONAL_INTERVAL_RECURRENCE',
                    'independent_verifier': 'PASS',
                }
                self.state.phase = 'M1_COMPLETE'
                final_checkpoint = self.checkpoint('M1 acceptance gates complete')
                paths = write_m1_report_package(
                    self.run_root, self.config, self.state, self.queue, self.test_report,
                    final_checkpoint, self.manifest,
                )
                self.logger.emit('m1_milestone_complete', run_id=self.state.run_id, reports=paths)
                return self._summary(f'M1 complete; report written to {paths["json"]}')
            predicted = self.queue.predicted_duration(item)
            if not self.guard.may_start(predicted):
                self.checkpoint('insufficient safe time for next M1 item')
                return self._summary('next M1 work item deferred to a fresh session')
            item.attempts += 1
            if item.attempts > self.config.max_item_attempts:
                item.status = 'failed'; item.last_error = 'Maximum M1 attempt count exceeded.'
                self.checkpoint('M1 work item exceeded attempt limit')
                raise RuntimeError(item.last_error)
            item.status = 'running'; self.checkpoint(f'before M1 item {item.phase}')
            started = time.monotonic()
            try:
                relative, digest = self._execute_item(item)
                item.result_relpath = relative; item.result_sha256 = digest; item.status = 'done'
                item.last_error = None; self.queue.record_timing(item.phase, time.monotonic() - started)
                if self.guard.state() is SessionState.RETURN:
                    return self._summary('hard return reached after atomic M1 done marker; resume will repair queue')
                self.checkpoint(f'after M1 item {item.phase}')
            except KeyboardInterrupt:
                item.status = 'pending'; item.last_error = 'KeyboardInterrupt'
                if self.guard.state() is not SessionState.RETURN:
                    self.checkpoint(f'interrupted M1 item {item.phase}')
                self._summary(f'KeyboardInterrupt in M1 item {item.phase}')
                raise
            except Exception as exc:
                item.status = 'failed' if item.attempts >= self.config.max_item_attempts else 'pending'
                item.last_error = f'{type(exc).__name__}: {exc}'
                if self.guard.state() is not SessionState.RETURN:
                    self.checkpoint(f'exception in M1 item {item.phase}')
                self._summary(f'exception in M1 item {item.phase}: {type(exc).__name__}')
                raise


def create_or_resume_m1(
    persistent_root: Path, config: M1Config, project_root: Path,
    run_id: str | None = None, test_report: dict[str, Any] | None = None,
) -> M1Orchestrator:
    acceptance_hash = _validate_acceptance_record(project_root, config)
    parent_hash = verify_accepted_m0_checkpoint(config)
    config_hash = config.config_hash(); source_hash = hash_tree(project_root / 'src')
    notebook_hash = _notebook_hash(project_root)
    document_hashes = governing_document_hashes(project_root)
    reference_hashes = reference_artifact_hashes(project_root)
    environment = environment_info(); runtime_signature = runtime_compatibility_signature(environment)
    convention_hash = sha256_bytes(canonical_json_bytes(CONVENTION))
    runs_root = persistent_root / 'runs'; runs_root.mkdir(parents=True, exist_ok=True)
    requested = run_id or os.environ.get('VALIDATED_RG_M1_RUN_ID')
    latest_pointer = persistent_root / 'LATEST_M1_RUN.json'
    if requested is None and latest_pointer.is_file():
        pointer = read_json(latest_pointer)
        if isinstance(pointer, dict) and pointer.get('config_hash') == config_hash:
            requested = pointer.get('run_id')
    if requested is None:
        requested = 'M1-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + config_hash[:12]
    if not isinstance(requested, str):
        raise M1CompatibilityError('M1 run ID must be a string.')
    safe_component(requested)
    if not requested.startswith('M1-'):
        raise M1CompatibilityError('M1 run ID must use the M1- namespace and cannot resume M0.')
    run_root = runs_root / requested; manifest_path = run_root / 'run_manifest.json'
    immutable_manifest_fields = {
        'milestone': 'M1', 'parent_milestone': config.parent_milestone,
        'parent_run_id': config.parent_run_id, 'parent_checkpoint': config.parent_checkpoint,
        'parent_checkpoint_path': config.parent_checkpoint_path,
        'parent_checkpoint_hash_manifest_sha256': parent_hash,
        'm0_acceptance_record_sha256': acceptance_hash, 'config_hash': config_hash,
        'source_hash': source_hash, 'convention_hash': convention_hash,
        'governing_document_hashes': document_hashes,
        'notebook_hash_policy': M1_NOTEBOOK_HASH_POLICY,
        'reference_artifact_hashes': reference_hashes,
        'runtime_compatibility': runtime_signature, 'certification_status': 'NOT_CERTIFIED',
    }
    if run_root.exists() and (run_root / 'run_config.json').is_file():
        if read_json(run_root / 'run_config.json') != config.canonical_payload():
            raise M1CompatibilityError('Immutable M1 run configuration differs.')
        if not manifest_path.is_file():
            raise M1CompatibilityError('Existing M1 run lacks run_manifest.json.')
        manifest = read_json(manifest_path)
        if not isinstance(manifest, dict) or any(manifest.get(key) != value for key, value in immutable_manifest_fields.items()):
            raise M1CompatibilityError('M1 manifest/source/parent/runtime identity changed.')
        manager = CheckpointManager(run_root, config, source_hash, notebook_hash)
        loaded = manager.load_latest(restore_rng=True)
        if loaded is None:
            raise M1CompatibilityError('Existing M1 run has no valid checkpoint.')
        repaired = loaded.queue.recover_interrupted(run_root)
        orchestrator = M1Orchestrator(
            persistent_root, run_root, project_root, config, loaded.state, loaded.queue,
            manager, test_report or {}, manifest,
        )
        if repaired:
            orchestrator.checkpoint(f'recovered {len(repaired)} interrupted M1 item(s)')
        print('Resumed M1 from:', loaded.path)
        return orchestrator
    if run_root.exists():
        raise M1CompatibilityError('M1 run directory exists but is incomplete; refusing overwrite.')
    run_root.mkdir(parents=True, exist_ok=False)
    for relative in ('logs', 'reports', 'artifacts', 'work_items', 'checkpoints'):
        (run_root / relative).mkdir(parents=True, exist_ok=True)
    manifest = {
        'schema_version': 1, 'run_id': requested, 'created_at': utc_now(),
        'notebook_hash': notebook_hash, 'environment': environment,
        'sector_ordering': 'ascending irrep dimension n',
        'rounding_policy': 'exact Fraction endpoints; outward Decimal display only',
        **immutable_manifest_fields,
    }
    atomic_write_json(run_root / 'run_config.json', config.canonical_payload())
    atomic_write_json(manifest_path, manifest)
    _seed_everything(config.seed)
    state = RunState(
        requested, config_hash, utc_now(), utc_now(), milestone='M1', phase='M1_BOOTSTRAP',
    )
    queue = WorkQueue()
    for phase in M1_PHASE_ORDER:
        queue.add(phase, config_hash, {'milestone': 'M1', 'phase': phase}, predicted_s=60.0)
    manager = CheckpointManager(run_root, config, source_hash, notebook_hash)
    orchestrator = M1Orchestrator(
        persistent_root, run_root, project_root, config, state, queue, manager,
        test_report or {}, manifest,
    )
    orchestrator.checkpoint('initial M1 run state')
    atomic_write_json(latest_pointer, {
        'milestone': 'M1', 'run_id': requested, 'config_hash': config_hash, 'updated_at': utc_now(),
    })
    return orchestrator


## Before running (Paperspace Gradient supported)

1. On Paperspace, upload the complete `validated_4d_su2_rg_codex_bundle` directory under `/notebooks`; the bootstrap locates it automatically. Elsewhere, place this notebook in durable project storage or set `VALIDATED_RG_PROJECT_ROOT`.
2. Mount durable checkpoint storage.
3. Set `VALIDATED_RG_PERSIST_ROOT` to a directory that survives runtime shutdown.
4. Explicitly acknowledge that property with `VALIDATED_RG_PERSIST_ACK=I_CONFIRM_THIS_PATH_IS_PERSISTENT`.
5. Confirm that the accepted M0 parent checkpoint `ckpt_000014` is still present and unchanged.
6. To resume a particular M1 run, set `VALIDATED_RG_M1_RUN_ID`; otherwise the durable `LATEST_M1_RUN.json` pointer is used when compatible.
7. Start from a fresh kernel and execute top to bottom.

On Paperspace, when both `/notebooks` and `/storage` exist, the bootstrap defaults to `/storage/validated_4d_su2_rg` and records the persistent-storage acknowledgement. Override `VALIDATED_RG_PERSIST_ROOT` before the bootstrap if the team needs an isolated subdirectory. Never run two kernels against the same persistent root.

Colab example after mounting Drive:

```python
import os
os.environ['VALIDATED_RG_PERSIST_ROOT'] = '/content/drive/MyDrive/validated_4d_su2_rg'
os.environ['VALIDATED_RG_PERSIST_ACK'] = 'I_CONFIRM_THIS_PATH_IS_PERSISTENT'
```

In [ ]:
from __future__ import annotations

import importlib.util
import os
import platform
import sys
from pathlib import Path

REQUIRED_PROJECT_FILES = (
    'validated_4d_su2_rg_gpu_driver.ipynb',
    'notebooks/00_project_setup.ipynb', 'notebooks/10_m0_accepted_audit.ipynb',
    'notebooks/20_m1_exact_2d.ipynb', 'notebooks/30_m2_armillary.ipynb',
    'notebooks/40_m3_gpu_triad_atrg.ipynb', 'notebooks/50_m4_derivatives.ipynb',
    'notebooks/60_m5_one_step_certificate.ipynb', 'notebooks/70_m6_multistep_certificate.ipynb',
    'validated_4d_su2_rg_codex_design.md', 'AGENTS.md', 'README.md',
    'audit/m0_accepted_parent.json',
    'requirements/paperspace.txt',
    'validated_4d_su2_rg_full_plan_bundle/validated_4d_su2_rg_codex_design_v0_2.md',
    'validated_4d_su2_rg_full_plan_bundle/M1_M6_VALIDATED_RG_ROADMAP.md',
    'validated_4d_su2_rg_full_plan_bundle/MATHEMATICAL_CERTIFICATION_SPEC.md',
    'validated_4d_su2_rg_full_plan_bundle/CODEX_PROMPTS_M1_M6.md',
    'validated_4d_su2_rg_full_plan_bundle/AGENTS_validated_4d_su2_rg_v0_2.md',
    'validated_4d_su2_rg_full_plan_bundle/validated_su2_rg_prototype.py',
    'validated_4d_su2_rg_full_plan_bundle/validated_su2_rg_report.md',
    'validated_4d_su2_rg_full_plan_bundle/validated_4d_su2_rg_M1_M6_tracker.ipynb',
)
BUNDLE_DIRECTORY_NAME = 'validated_4d_su2_rg_codex_bundle'
PERSIST_ACK_TOKEN = 'I_CONFIRM_THIS_PATH_IS_PERSISTENT'
is_paperspace = Path('/notebooks').is_dir() and Path('/storage').is_dir()
explicit_project_root = os.environ.get('VALIDATED_RG_PROJECT_ROOT')
if explicit_project_root:
    project_candidates = [Path(explicit_project_root)]
else:
    project_candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd() / BUNDLE_DIRECTORY_NAME,
        Path.cwd().parent / BUNDLE_DIRECTORY_NAME,
        Path('/notebooks'),
        Path('/notebooks') / BUNDLE_DIRECTORY_NAME,
        Path('/storage') / BUNDLE_DIRECTORY_NAME,
    ]
resolved_candidates = list(dict.fromkeys(candidate.expanduser().resolve() for candidate in project_candidates))
PROJECT_ROOT = next((
    candidate for candidate in resolved_candidates
    if all((candidate / name).is_file() for name in REQUIRED_PROJECT_FILES)
), resolved_candidates[0])
missing_project_files = [name for name in REQUIRED_PROJECT_FILES if not (PROJECT_ROOT / name).is_file()]
if missing_project_files:
    raise RuntimeError(
        f'Project root is missing required M0/full-plan files: {missing_project_files}. '
        f'Tried: {[str(path) for path in resolved_candidates]}. '
        'Upload the complete bundle or set VALIDATED_RG_PROJECT_ROOT.'
    )
os.environ.setdefault('VALIDATED_RG_PROJECT_ROOT', str(PROJECT_ROOT))
if is_paperspace:
    os.environ.setdefault('VALIDATED_RG_PERSIST_ROOT', '/storage/validated_4d_su2_rg')
    os.environ.setdefault('VALIDATED_RG_PERSIST_ACK', PERSIST_ACK_TOKEN)
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
for relative in ('src', 'tests'):
    (PROJECT_ROOT / relative).mkdir(parents=True, exist_ok=True)
if sys.version_info < (3, 11):
    raise RuntimeError(f'Python 3.11+ is required; found {sys.version.split()[0]}.')
missing = [name for name in ('numpy', 'pytest') if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(f'Missing required packages: {missing}. Install them in this runtime, then restart the kernel.')
print('Project root:', PROJECT_ROOT)
print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())
print('Paperspace Gradient detected:', is_paperspace)
if is_paperspace:
    print('Paperspace persistent root:', os.environ['VALIDATED_RG_PERSIST_ROOT'])
print('Module source cells below are authoritative and overwrite generated src/tests files.')

In [ ]:
%%writefile src/__init__.py
'''M0 persistence plus M1 exact-benchmark package for validated 4D SU(2) RG.'''

IMPLEMENTED_MILESTONES = ('M0', 'M1')
ACTIVE_MILESTONE = 'M1'
CERTIFICATION_STATUS = 'NOT_CERTIFIED'


In [ ]:
%%writefile src/config.py
from __future__ import annotations

import hashlib
import json
import math
from dataclasses import asdict, dataclass
from typing import Any


class ConfigError(ValueError):
    '''Raised when an immutable run configuration violates M0 safety rules.'''


@dataclass(frozen=True, slots=True)
class RunConfig:
    schema_version: int = 1
    project_name: str = 'validated_4d_su2_rg'
    milestone: str = 'M0'
    seed: int = 20260719
    checkpoint_interval_s: float = 15.0 * 60.0
    max_work_item_s: float = 20.0 * 60.0
    short_task_limit_s: float = 5.0 * 60.0
    checkpoint_reserve_s: float = 10.0 * 60.0
    no_long_task_after_s: float = 5.0 * 3600.0
    drain_after_s: float = 5.0 * 3600.0 + 15.0 * 60.0
    final_save_after_s: float = 5.0 * 3600.0 + 20.0 * 60.0
    hard_return_s: float = 5.0 * 3600.0 + 30.0 * 60.0
    tensor_shard_bytes: int = 256 * 1024 * 1024
    dummy_items: int = 6
    dummy_size: int = 128
    dummy_steps: int = 2
    dummy_predicted_s: float = 30.0
    max_item_attempts: int = 3
    prefer_cuda: bool = True
    certification_status: str = 'NOT_CERTIFIED'

    def __post_init__(self) -> None:
        if self.schema_version != 1 or self.project_name != 'validated_4d_su2_rg':
            raise ConfigError('Unsupported M0 schema or project identity.')
        if self.milestone != 'M0':
            raise ConfigError('This notebook implements M0 only.')
        if self.certification_status != 'NOT_CERTIFIED':
            raise ConfigError('M0 certification_status is immutable and must be NOT_CERTIFIED.')
        if not isinstance(self.prefer_cuda, bool):
            raise ConfigError('prefer_cuda must be a boolean execution policy.')
        integer_fields = (
            self.seed, self.tensor_shard_bytes, self.dummy_items, self.dummy_size,
            self.dummy_steps, self.max_item_attempts,
        )
        if any(not isinstance(value, int) or isinstance(value, bool) for value in integer_fields):
            raise ConfigError('Seed, shard, dummy, and attempt fields must be integers.')
        schedule = (
            0.0,
            self.no_long_task_after_s,
            self.drain_after_s,
            self.final_save_after_s,
            self.hard_return_s,
        )
        duration_fields = schedule[1:] + (
            self.checkpoint_interval_s, self.max_work_item_s, self.short_task_limit_s,
            self.checkpoint_reserve_s, self.dummy_predicted_s,
        )
        if any(not isinstance(value, (int, float)) or isinstance(value, bool) for value in duration_fields):
            raise ConfigError('Session and prediction durations must be real numbers.')
        if any(not math.isfinite(value) for value in schedule):
            raise ConfigError('Session schedule must be finite.')
        if tuple(sorted(schedule)) != schedule or len(set(schedule)) != len(schedule):
            raise ConfigError('Session thresholds must be strictly increasing.')
        if self.hard_return_s > 5.5 * 3600.0:
            raise ConfigError('Hard return may not exceed 5 h 30 min.')
        if self.final_save_after_s > 5.0 * 3600.0 + 20.0 * 60.0:
            raise ConfigError('Final checkpointing must begin no later than 5 h 20 min.')
        if not 0.0 < self.max_work_item_s <= 20.0 * 60.0:
            raise ConfigError('A work item may not exceed 20 minutes.')
        if self.checkpoint_interval_s <= 0.0 or self.checkpoint_interval_s > 15.0 * 60.0:
            raise ConfigError('Checkpoint interval may not exceed 15 minutes.')
        positive_durations = (
            self.max_work_item_s, self.short_task_limit_s, self.checkpoint_reserve_s,
            self.checkpoint_interval_s, self.dummy_predicted_s,
        )
        if any(not math.isfinite(value) or value <= 0.0 for value in positive_durations):
            raise ConfigError('Durations must be positive and finite.')
        if self.short_task_limit_s > self.max_work_item_s:
            raise ConfigError('Short-task limit may not exceed the work-item limit.')
        if self.dummy_predicted_s > self.max_work_item_s:
            raise ConfigError('Dummy prediction may not exceed the work-item limit.')
        if self.tensor_shard_bytes <= 0 or self.dummy_items < 0:
            raise ConfigError('Shard size and dummy item count are invalid.')
        if self.dummy_size <= 0 or self.dummy_steps <= 0 or self.max_item_attempts <= 0:
            raise ConfigError('Dummy dimensions, steps, and attempt count must be positive.')
        if self.dummy_size > 512 or self.dummy_steps > 8 or self.dummy_items > 10_000:
            raise ConfigError('M0 dummy workload exceeds the bounded safety caps.')
        if not 0 <= self.seed < 2**32:
            raise ConfigError('Seed must be accepted by NumPy RandomState (0 <= seed < 2**32).')

    def canonical_payload(self) -> dict[str, Any]:
        return asdict(self)

    def canonical_json(self) -> str:
        return json.dumps(self.canonical_payload(), sort_keys=True, separators=(',', ':'), allow_nan=False)

    def config_hash(self) -> str:
        return hashlib.sha256(self.canonical_json().encode('utf-8')).hexdigest()


In [ ]:
%%writefile src/common.py
from __future__ import annotations

import errno
import hashlib
import json
import os
import uuid
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable


def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


def canonical_json_bytes(payload: Any) -> bytes:
    return json.dumps(payload, ensure_ascii=False, sort_keys=True, separators=(',', ':'), allow_nan=False).encode('utf-8')


def sha256_bytes(payload: bytes) -> str:
    return hashlib.sha256(payload).hexdigest()


def sha256_file(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def fsync_descriptor(descriptor: int) -> None:
    try:
        os.fsync(descriptor)
    except OSError as exc:
        not_supported = getattr(errno, 'ENOTSUP', errno.EINVAL)
        unsupported = {errno.EINVAL, not_supported, getattr(errno, 'EOPNOTSUPP', not_supported)}
        if exc.errno not in unsupported:
            raise


def fsync_directory(path: Path) -> None:
    try:
        descriptor = os.open(path, os.O_RDONLY)
    except OSError:
        return
    try:
        fsync_descriptor(descriptor)
    finally:
        os.close(descriptor)


def fsync_file(path: Path) -> None:
    with path.open('rb') as handle:
        fsync_descriptor(handle.fileno())


def atomic_write_bytes(path: Path, payload: bytes) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f'.{path.name}.tmp-{uuid.uuid4().hex}')
    with temporary.open('wb') as handle:
        handle.write(payload)
        handle.flush()
        fsync_descriptor(handle.fileno())
    os.replace(temporary, path)
    fsync_directory(path.parent)


def atomic_write_text(path: Path, payload: str) -> None:
    atomic_write_bytes(path, payload.encode('utf-8'))


def atomic_write_json(path: Path, payload: Any) -> None:
    atomic_write_bytes(path, canonical_json_bytes(payload))


def read_json(path: Path) -> Any:
    return json.loads(path.read_text(encoding='utf-8'))


def safe_component(value: str) -> str:
    if not value or value in {'.', '..'} or '/' in value or '\\' in value:
        raise ValueError(f'Unsafe path component: {value!r}')
    return value


def hash_tree(root: Path, suffixes: Iterable[str] = ('.py',)) -> str:
    allowed = set(suffixes)
    digest = hashlib.sha256()
    for path in sorted(p for p in root.rglob('*') if p.is_file() and p.suffix in allowed and '__pycache__' not in p.parts):
        relative = path.relative_to(root).as_posix().encode('utf-8')
        digest.update(len(relative).to_bytes(8, 'big'))
        digest.update(relative)
        file_hash = sha256_file(path).encode('ascii')
        digest.update(file_hash)
    return digest.hexdigest()


def directory_size(path: Path) -> int:
    return sum(item.stat().st_size for item in path.rglob('*') if item.is_file())


In [ ]:
%%writefile src/runtime.py
from __future__ import annotations

import os
import platform
import sys
import uuid
from pathlib import Path
from typing import Any

import numpy as np

from .common import atomic_write_json, fsync_descriptor, fsync_directory, read_json, utc_now

try:
    import torch
except ImportError:
    torch = None

PERSIST_ROOT_ENV = 'VALIDATED_RG_PERSIST_ROOT'
PERSIST_ACK_ENV = 'VALIDATED_RG_PERSIST_ACK'
PERSIST_ACK_TOKEN = 'I_CONFIRM_THIS_PATH_IS_PERSISTENT'


class PersistentRootError(RuntimeError):
    '''Raised when durable storage has not been explicitly established.'''


def _is_relative_to(path: Path, parent: Path) -> bool:
    try:
        path.relative_to(parent)
        return True
    except ValueError:
        return False


def validate_persistent_root(raw: str | None = None, acknowledgement: str | None = None) -> Path:
    raw = raw if raw is not None else os.environ.get(PERSIST_ROOT_ENV)
    acknowledgement = acknowledgement if acknowledgement is not None else os.environ.get(PERSIST_ACK_ENV)
    if not raw:
        raise PersistentRootError(f'{PERSIST_ROOT_ENV} is required; no computation was started.')
    if acknowledgement != PERSIST_ACK_TOKEN:
        raise PersistentRootError(
            f'Set {PERSIST_ACK_ENV}={PERSIST_ACK_TOKEN} only after confirming shutdown persistence.'
        )
    root = Path(raw).expanduser().resolve()
    forbidden = (
        Path('/tmp'), Path('/var/tmp'), Path('/dev/shm'), Path('/private/tmp'),
        Path('/private/var/folders'),
    )
    if root == Path('/') or any(root == item or _is_relative_to(root, item) for item in forbidden):
        raise PersistentRootError(f'Ephemeral or unsafe checkpoint root rejected: {root}')
    if root == Path('/content') or (_is_relative_to(root, Path('/content')) and not (
        _is_relative_to(root, Path('/content/drive')) or _is_relative_to(root, Path('/content/gdrive'))
    )):
        raise PersistentRootError('/content local storage is ephemeral; use a mounted Drive path.')
    root.mkdir(parents=True, exist_ok=True)
    probe = root / f'.write-probe-{uuid.uuid4().hex}'
    payload = utc_now().encode('utf-8')
    with probe.open('wb') as handle:
        handle.write(payload)
        handle.flush()
        fsync_descriptor(handle.fileno())
    if probe.read_bytes() != payload:
        raise PersistentRootError('Persistent-root write/read probe failed.')
    probe.unlink()
    fsync_directory(root)
    marker = root / '.validated_rg_persistent_root.json'
    if marker.exists():
        stored = read_json(marker)
        if stored.get('resolved_root') != str(root):
            raise PersistentRootError('Persistent-root marker does not match the resolved path.')
    else:
        atomic_write_json(marker, {'schema_version': 1, 'resolved_root': str(root), 'acknowledged_at': utc_now()})
    for relative in (
        'project', 'cache/wigner', 'cache/fusion', 'cache/armillary',
        'cache/contraction_paths',
    ):
        (root / relative).mkdir(parents=True, exist_ok=True)
    fsync_directory(root)
    return root


def configure_numerics() -> None:
    if torch is not None and torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False


def environment_info() -> dict[str, Any]:
    configure_numerics()
    cuda_available = bool(torch is not None and torch.cuda.is_available())
    info: dict[str, Any] = {
        'captured_at': utc_now(),
        'python': sys.version,
        'platform': platform.platform(),
        'numpy': np.__version__,
        'torch': getattr(torch, '__version__', None),
        'cuda_available': cuda_available,
        'cuda_runtime': getattr(getattr(torch, 'version', None), 'cuda', None) if torch is not None else None,
        'cudnn': torch.backends.cudnn.version() if cuda_available else None,
        'tf32_matmul': torch.backends.cuda.matmul.allow_tf32 if cuda_available else None,
        'tf32_cudnn': torch.backends.cudnn.allow_tf32 if cuda_available else None,
        'execution_environment': (
            'paperspace_gradient'
            if Path('/notebooks').is_dir() and Path('/storage').is_dir()
            else 'generic'
        ),
        'paperspace_notebooks_mount': Path('/notebooks').is_dir(),
        'paperspace_storage_mount': Path('/storage').is_dir(),
        'configured_persistent_root': os.environ.get(PERSIST_ROOT_ENV),
    }
    if cuda_available:
        index = torch.cuda.current_device()
        properties = torch.cuda.get_device_properties(index)
        info.update({
            'gpu_index': index,
            'gpu_name': properties.name,
            'gpu_total_memory': properties.total_memory,
            'gpu_capability': list(torch.cuda.get_device_capability(index)),
            'gpu_count': torch.cuda.device_count(),
        })
    return info


RUNTIME_COMPATIBILITY_KEYS = (
    'python', 'platform', 'numpy', 'torch', 'cuda_available', 'cuda_runtime', 'cudnn',
    'tf32_matmul', 'tf32_cudnn', 'execution_environment', 'gpu_name',
    'gpu_capability', 'gpu_count',
)


def runtime_compatibility_signature(info: dict[str, Any] | None = None) -> dict[str, Any]:
    captured = environment_info() if info is None else info
    return {key: captured.get(key) for key in RUNTIME_COMPATIBILITY_KEYS}


In [ ]:
%%writefile src/session_guard.py
from __future__ import annotations

import math
import time
from dataclasses import dataclass, field
from enum import Enum
from typing import Callable

from .config import RunConfig


class SessionState(str, Enum):
    RUN = 'RUN'
    NO_LONG_TASK = 'NO_LONG_TASK'
    DRAIN = 'DRAIN'
    FINAL_SAVE = 'FINAL_SAVE'
    RETURN = 'RETURN'


@dataclass(slots=True)
class SessionGuard:
    config: RunConfig
    clock: Callable[[], float] = time.monotonic
    started_monotonic: float = field(init=False)
    last_checkpoint_monotonic: float = field(init=False)

    def __post_init__(self) -> None:
        now = self.clock()
        self.started_monotonic = now
        self.last_checkpoint_monotonic = now

    def elapsed_s(self) -> float:
        return max(0.0, self.clock() - self.started_monotonic)

    def remaining_s(self) -> float:
        return max(0.0, self.config.hard_return_s - self.elapsed_s())

    def state(self) -> SessionState:
        elapsed = self.elapsed_s()
        if elapsed >= self.config.hard_return_s:
            return SessionState.RETURN
        if elapsed >= self.config.final_save_after_s:
            return SessionState.FINAL_SAVE
        if elapsed >= self.config.drain_after_s:
            return SessionState.DRAIN
        if elapsed >= self.config.no_long_task_after_s:
            return SessionState.NO_LONG_TASK
        return SessionState.RUN

    def checkpoint_due(self) -> bool:
        return self.clock() - self.last_checkpoint_monotonic >= self.config.checkpoint_interval_s

    def mark_checkpoint(self) -> None:
        self.last_checkpoint_monotonic = self.clock()

    def may_start(self, predicted_s: float) -> bool:
        if not math.isfinite(predicted_s) or predicted_s <= 0.0:
            return False
        state = self.state()
        if state not in {SessionState.RUN, SessionState.NO_LONG_TASK}:
            return False
        if predicted_s > self.config.max_work_item_s:
            return False
        if state is SessionState.NO_LONG_TASK and predicted_s > self.config.short_task_limit_s:
            return False
        return self.remaining_s() >= 1.3 * predicted_s + self.config.checkpoint_reserve_s


In [ ]:
%%writefile src/work_queue.py
from __future__ import annotations

import math
from dataclasses import asdict, dataclass, field
from pathlib import Path
from statistics import quantiles
from typing import Any, Literal

from .common import canonical_json_bytes, read_json, sha256_bytes, sha256_file

WorkStatus = Literal['pending', 'running', 'done', 'failed', 'blocked_resource']


@dataclass(slots=True)
class WorkItem:
    item_id: str
    phase: str
    input_hash: str
    parameters: dict[str, Any]
    status: WorkStatus = 'pending'
    attempts: int = 0
    predicted_s: float = 60.0
    result_relpath: str | None = None
    result_sha256: str | None = None
    last_error: str | None = None


@dataclass(slots=True)
class WorkQueue:
    items: dict[str, WorkItem] = field(default_factory=dict)
    timings_by_phase: dict[str, list[float]] = field(default_factory=dict)

    @staticmethod
    def make_id(phase: str, input_hash: str, parameters: dict[str, Any]) -> str:
        payload = {'phase': phase, 'input_hash': input_hash, 'parameters': parameters}
        return sha256_bytes(canonical_json_bytes(payload))

    def add(self, phase: str, input_hash: str, parameters: dict[str, Any], predicted_s: float) -> str:
        if not math.isfinite(predicted_s) or predicted_s <= 0.0:
            raise ValueError('predicted_s must be positive and finite.')
        item_id = self.make_id(phase, input_hash, parameters)
        candidate = WorkItem(item_id, phase, input_hash, parameters, predicted_s=predicted_s)
        existing = self.items.setdefault(item_id, candidate)
        if existing.phase != phase or existing.input_hash != input_hash or existing.parameters != parameters:
            raise RuntimeError('Content-address collision with unequal work specification.')
        return item_id

    def next_pending(self) -> WorkItem | None:
        for item_id in sorted(self.items):
            item = self.items[item_id]
            if item.status == 'pending':
                return item
        return None

    def predicted_duration(self, item: WorkItem) -> float:
        samples = self.timings_by_phase.get(item.phase, [])
        if len(samples) < 2:
            return item.predicted_s
        if len(samples) < 20:
            return max(item.predicted_s, max(samples))
        p95 = quantiles(samples, n=20, method='inclusive')[18]
        return max(item.predicted_s, p95)

    def record_timing(self, phase: str, elapsed_s: float) -> None:
        if math.isfinite(elapsed_s) and elapsed_s >= 0.0:
            values = self.timings_by_phase.setdefault(phase, [])
            values.append(elapsed_s)
            del values[:-100]

    def validate(self) -> None:
        allowed = {'pending', 'running', 'done', 'failed', 'blocked_resource'}
        for key, item in self.items.items():
            if not all(isinstance(value, str) for value in (key, item.item_id, item.phase, item.input_hash)):
                raise ValueError('Work item identity fields must be strings.')
            if not isinstance(item.parameters, dict):
                raise ValueError(f'Work item parameters must be a mapping: {key}')
            if key != item.item_id or key != self.make_id(item.phase, item.input_hash, item.parameters):
                raise ValueError(f'Work item content hash mismatch: {key}')
            if item.status not in allowed:
                raise ValueError(f'Unknown work item status: {item.status!r}')
            if not isinstance(item.attempts, int) or isinstance(item.attempts, bool) or item.attempts < 0:
                raise ValueError(f'Invalid work item attempt count: {key}')
            if not isinstance(item.predicted_s, (int, float)) or not math.isfinite(item.predicted_s) or item.predicted_s <= 0.0:
                raise ValueError(f'Invalid work item counters or prediction: {key}')
            if item.status == 'done' and (not item.result_relpath or not item.result_sha256):
                raise ValueError(f'Done work item lacks result metadata: {key}')
            if item.result_relpath is not None and not isinstance(item.result_relpath, str):
                raise ValueError(f'Invalid work item result path: {key}')
            if item.last_error is not None and not isinstance(item.last_error, str):
                raise ValueError(f'Invalid work item error field: {key}')
            if item.result_sha256 is not None and (
                not isinstance(item.result_sha256, str) or len(item.result_sha256) != 64
                or any(character not in '0123456789abcdef' for character in item.result_sha256)
            ):
                raise ValueError(f'Invalid work item result hash: {key}')
        for phase, samples in self.timings_by_phase.items():
            if not isinstance(phase, str) or any(not math.isfinite(value) or value < 0.0 for value in samples):
                raise ValueError(f'Invalid timing history for phase {phase!r}.')

    def recover_interrupted(self, run_root: Path) -> list[str]:
        repaired: list[str] = []
        marker_root = run_root / 'work_items'
        for item in self.items.values():
            if item.status not in {'running', 'done'}:
                continue
            previous_status = item.status
            marker = marker_root / f'{item.item_id}.done'
            valid = False
            if marker.is_file():
                try:
                    payload = read_json(marker)
                    relative = payload['result_relpath']
                    digest = payload['result_sha256']
                    if payload.get('item_id') != item.item_id:
                        raise ValueError('Done marker item_id mismatch.')
                    if not isinstance(relative, str) or not isinstance(digest, str):
                        raise ValueError('Done marker has invalid field types.')
                    result = (run_root / relative).resolve()
                    result.relative_to(run_root.resolve())
                    valid = result.is_file() and sha256_file(result) == digest
                    if previous_status == 'done' and (
                        item.result_relpath != relative or item.result_sha256 != digest
                    ):
                        valid = False
                    if valid:
                        item.result_relpath = relative
                        item.result_sha256 = digest
                except (KeyError, OSError, TypeError, ValueError):
                    valid = False
            item.status = 'done' if valid else 'pending'
            if not valid:
                item.result_relpath = None
                item.result_sha256 = None
            if previous_status == 'running' or item.status != previous_status:
                repaired.append(item.item_id)
        return repaired

    def to_payload(self) -> dict[str, Any]:
        return {
            'items': {key: asdict(value) for key, value in sorted(self.items.items())},
            'timings_by_phase': self.timings_by_phase,
        }

    @classmethod
    def from_payload(cls, payload: dict[str, Any]) -> 'WorkQueue':
        queue = cls(
            items={key: WorkItem(**value) for key, value in payload['items'].items()},
            timings_by_phase={key: [float(v) for v in values] for key, values in payload.get('timings_by_phase', {}).items()},
        )
        queue.validate()
        return queue


In [ ]:
%%writefile src/checkpoint.py
from __future__ import annotations

import os
import pickle
import random
import shutil
import sys
import time
import uuid
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Protocol

import numpy as np

from .common import (
    atomic_write_json, atomic_write_text, directory_size, fsync_descriptor, fsync_directory,
    fsync_file, read_json,
    safe_component, sha256_file, utc_now,
)
from .runtime import environment_info
from .work_queue import WorkQueue

try:
    import torch
except ImportError:
    torch = None


class CheckpointError(RuntimeError):
    '''Raised when checkpoint creation, validation, or restoration fails.'''


class ConfigMismatchError(CheckpointError):
    '''Raised when a checkpoint belongs to incompatible immutable inputs.'''


class CheckpointConfig(Protocol):
    tensor_shard_bytes: int

    def config_hash(self) -> str: ...

    def canonical_payload(self) -> dict[str, Any]: ...


@dataclass(slots=True)
class RunState:
    run_id: str
    config_hash: str
    created_at: str
    updated_at: str
    milestone: str = 'M0'
    phase: str = 'BOOTSTRAP'
    subphase: str = ''
    rg_step: int = 0
    direction: str = ''
    checkpoint_index: int = 0
    certification_status: str = 'NOT_CERTIFIED'
    bounds: dict[str, str] = field(default_factory=dict)
    normalization_log: list[str] = field(default_factory=list)
    notes: list[str] = field(default_factory=list)

    def assert_safe(self) -> None:
        if self.milestone == 'M0':
            self.assert_m0_safe()
        elif self.milestone == 'M1':
            self.assert_m1_safe()
        else:
            raise CheckpointError(f'Unsupported milestone in checkpoint state: {self.milestone!r}')

    def _assert_common_safe(self) -> None:
        if self.certification_status != 'NOT_CERTIFIED':
            raise CheckpointError('Checkpoint state attempted to leave NOT_CERTIFIED.')
        safe_component(self.run_id)
        if not isinstance(self.checkpoint_index, int) or self.checkpoint_index < 0:
            raise CheckpointError('Checkpoint index must be a nonnegative integer.')
        if not all(isinstance(value, str) for value in (self.config_hash, self.created_at, self.updated_at)):
            raise CheckpointError('State identity/timestamp fields must be strings.')
        if len(self.config_hash) != 64 or any(character not in '0123456789abcdef' for character in self.config_hash):
            raise CheckpointError('State config hash is malformed.')
        if not all(isinstance(value, str) for value in self.notes + self.normalization_log):
            raise CheckpointError('State notes and normalization log must contain strings only.')

    def assert_m0_safe(self) -> None:
        self._assert_common_safe()
        if self.milestone != 'M0':
            raise CheckpointError('M0 assertion received a non-M0 state.')
        if self.phase not in {'BOOTSTRAP', 'DUMMY', 'M0_COMPLETE'}:
            raise CheckpointError(f'Phase {self.phase!r} is outside M0.')
        if self.rg_step != 0 or self.direction or self.subphase:
            raise CheckpointError('M0 state may not contain RG progression fields.')
        if self.bounds:
            raise CheckpointError('M0 constructs no rigorous bounds; bounds must remain absent.')

    def assert_m1_safe(self) -> None:
        self._assert_common_safe()
        if self.milestone != 'M1':
            raise CheckpointError('M1 assertion received a non-M1 state.')
        if self.phase not in {'M1_BOOTSTRAP', 'M1_RUNNING', 'M1_COMPLETE'}:
            raise CheckpointError(f'Phase {self.phase!r} is outside M1.')
        if self.rg_step != 0 or self.direction or self.subphase:
            raise CheckpointError('M1 state may not contain 4D RG progression fields.')
        if not isinstance(self.bounds, dict) or any(
            not isinstance(key, str) or not isinstance(value, str)
            for key, value in self.bounds.items()
        ):
            raise CheckpointError('M1 bound provenance must be a string mapping.')


@dataclass(frozen=True, slots=True)
class CheckpointSaveResult:
    path: Path
    index: int
    size_bytes: int
    save_s: float
    verify_s: float


@dataclass(frozen=True, slots=True)
class LoadedCheckpoint:
    state: RunState
    queue: WorkQueue
    path: Path
    tensors: dict[str, Any]
    skipped_invalid: tuple[str, ...]


def _rng_runtime_signature() -> dict[str, Any]:
    cuda_available = bool(torch is not None and torch.cuda.is_available())
    return {
        'python': sys.version,
        'numpy': np.__version__,
        'torch': getattr(torch, '__version__', None),
        'cuda_runtime': getattr(getattr(torch, 'version', None), 'cuda', None) if torch is not None else None,
        'cuda_available': cuda_available,
        'cuda_device_count': torch.cuda.device_count() if cuda_available else 0,
    }


def capture_rng_state() -> dict[str, Any]:
    payload: dict[str, Any] = {
        'python': random.getstate(), 'numpy': np.random.get_state(),
        'runtime_signature': _rng_runtime_signature(),
    }
    if torch is not None:
        payload['torch_cpu'] = torch.get_rng_state().cpu()
        if torch.cuda.is_available():
            payload['torch_cuda'] = [state.cpu() for state in torch.cuda.get_rng_state_all()]
    return payload


def restore_rng_state(payload: dict[str, Any]) -> None:
    if not isinstance(payload, dict) or 'python' not in payload or 'numpy' not in payload:
        raise CheckpointError('RNG checkpoint is missing Python or NumPy state.')
    if payload.get('runtime_signature') != _rng_runtime_signature():
        raise CheckpointError('RNG runtime signature changed; exact restoration is impossible.')
    saved_with_torch = 'torch_cpu' in payload
    if saved_with_torch != (torch is not None):
        raise CheckpointError('PyTorch availability changed; exact RNG restoration is impossible.')
    saved_with_cuda = 'torch_cuda' in payload
    cuda_available = bool(torch is not None and torch.cuda.is_available())
    if saved_with_cuda != cuda_available:
        raise CheckpointError('CUDA availability changed; exact RNG restoration is impossible.')
    random.setstate(payload['python'])
    np.random.set_state(payload['numpy'])
    if torch is not None:
        torch.set_rng_state(payload['torch_cpu'])
        if cuda_available:
            if len(payload['torch_cuda']) != torch.cuda.device_count():
                raise CheckpointError('CUDA RNG device count changed; refusing partial restoration.')
            torch.cuda.set_rng_state_all(payload['torch_cuda'])


def _write_pickle(path: Path, payload: Any) -> None:
    with path.open('wb') as handle:
        pickle.dump(payload, handle, protocol=pickle.HIGHEST_PROTOCOL)
        handle.flush()
        fsync_descriptor(handle.fileno())


class TensorShardStore:
    def __init__(self, shard_bytes: int) -> None:
        if shard_bytes <= 0:
            raise ValueError('shard_bytes must be positive.')
        self.shard_bytes = shard_bytes

    def save(self, root: Path, tensors: dict[str, Any]) -> dict[str, Any]:
        root.mkdir(parents=True, exist_ok=False)
        index: dict[str, Any] = {}
        for name, value in sorted(tensors.items()):
            safe_component(name)
            is_torch = torch is not None and isinstance(value, torch.Tensor)
            if is_torch:
                array = value.detach().cpu().contiguous()
                shape = tuple(array.shape)
                item_size = array.element_size()
                kind = 'torch'
                dtype = str(array.dtype)
            elif isinstance(value, np.ndarray):
                array = np.array(value, copy=True, order='C', subok=False)
                shape = tuple(array.shape)
                item_size = array.dtype.itemsize
                kind = 'numpy'
                dtype = str(array.dtype)
            else:
                raise TypeError(f'Unsupported tensor type for {name}: {type(value)!r}')
            flat = array.reshape(-1)
            element_count = int(flat.numel()) if kind == 'torch' else int(flat.size)
            elements_per_shard = max(1, self.shard_bytes // max(1, item_size))
            files: list[str] = []
            starts = (0,) if element_count == 0 else range(0, element_count, elements_per_shard)
            for shard_index, start in enumerate(starts):
                stop = min(element_count, start + elements_per_shard)
                shard = flat[start:stop]
                suffix = '.pt' if kind == 'torch' else '.npy'
                filename = f'{name}.shard-{shard_index:06d}{suffix}'
                path = root / filename
                if kind == 'torch':
                    torch.save(shard, path)
                    fsync_file(path)
                else:
                    with path.open('wb') as handle:
                        np.save(handle, shard, allow_pickle=False)
                        handle.flush()
                        fsync_descriptor(handle.fileno())
                files.append(filename)
            index[name] = {'kind': kind, 'dtype': dtype, 'shape': list(shape), 'files': files}
        atomic_write_json(root / 'index.json', index)
        return index

    def load(self, root: Path) -> dict[str, Any]:
        index = read_json(root / 'index.json')
        if not isinstance(index, dict):
            raise CheckpointError('Tensor index must be a mapping.')
        result: dict[str, Any] = {}
        for name, metadata in index.items():
            safe_component(name)
            if not isinstance(metadata, dict):
                raise CheckpointError(f'Tensor metadata must be a mapping: {name}')
            shape = metadata.get('shape')
            if not isinstance(shape, list) or any(
                not isinstance(size, int) or isinstance(size, bool) or size < 0 for size in shape
            ):
                raise CheckpointError(f'Invalid tensor shape metadata: {name}')
            kind = metadata.get('kind')
            if kind not in {'torch', 'numpy'} or not isinstance(metadata.get('dtype'), str):
                raise CheckpointError(f'Invalid tensor kind or dtype metadata: {name}')
            raw_files = metadata['files']
            if not isinstance(raw_files, list) or not raw_files:
                raise CheckpointError(f'Tensor {name} has no shard files.')
            if any(not isinstance(filename, str) for filename in raw_files):
                raise CheckpointError(f'Tensor {name} has a non-string shard name.')
            if len(set(raw_files)) != len(raw_files) or (shape == [] and len(raw_files) != 1):
                raise CheckpointError(f'Tensor {name} has invalid or duplicate shard entries.')
            files: list[Path] = []
            for shard_index, filename in enumerate(raw_files):
                safe_component(filename)
                suffix = '.pt' if kind == 'torch' else '.npy'
                expected_filename = f'{name}.shard-{shard_index:06d}{suffix}'
                if filename != expected_filename:
                    raise CheckpointError(f'Tensor {name} has a mismatched shard filename.')
                files.append(root / filename)
            if kind == 'torch':
                if torch is None:
                    raise CheckpointError(f'PyTorch is required to load tensor {name}.')
                try:
                    chunks = [torch.load(path, map_location='cpu', weights_only=True) for path in files]
                except TypeError:  # PyTorch versions predating weights_only
                    chunks = [torch.load(path, map_location='cpu') for path in files]
                if any(not isinstance(chunk, torch.Tensor) for chunk in chunks):
                    raise CheckpointError(f'Non-tensor object found in shards: {name}')
                flat = torch.cat(chunks, dim=0)
                value = flat.reshape(tuple(shape))
                if list(value.shape) != shape:
                    raise CheckpointError(f'Sharded tensor shape mismatch: {name}')
                if str(value.dtype) != metadata['dtype']:
                    raise CheckpointError(f'Sharded tensor dtype mismatch: {name}')
            elif kind == 'numpy':
                chunks = [np.load(path, allow_pickle=False) for path in files]
                flat = np.concatenate(chunks, axis=0)
                value = flat.reshape(tuple(shape))
                if list(value.shape) != shape:
                    raise CheckpointError(f'Sharded array shape mismatch: {name}')
                if str(value.dtype) != metadata['dtype']:
                    raise CheckpointError(f'Sharded array dtype mismatch: {name}')
            result[name] = value
        return result


class CheckpointManager:
    def __init__(self, run_root: Path, config: CheckpointConfig, source_hash: str, notebook_hash: str | None) -> None:
        self.run_root = run_root
        self.config = config
        self.source_hash = source_hash
        self.notebook_hash = notebook_hash
        self.checkpoint_root = run_root / 'checkpoints'
        self.checkpoint_root.mkdir(parents=True, exist_ok=True)
        self.tensor_store = TensorShardStore(config.tensor_shard_bytes)

    def _candidate_paths(self) -> list[Path]:
        candidates: list[tuple[int, Path]] = []
        for path in self.checkpoint_root.glob('ckpt_*'):
            if not path.is_dir():
                continue
            try:
                index = int(path.name.removeprefix('ckpt_'))
            except ValueError:
                continue
            candidates.append((index, path))
        return [path for _, path in sorted(candidates, reverse=True)]

    def _next_index(self, state: RunState) -> int:
        existing = [int(path.name.removeprefix('ckpt_')) for path in self._candidate_paths()]
        return max([state.checkpoint_index, *existing], default=0) + 1

    def save(self, state: RunState, queue: WorkQueue, tensors: dict[str, Any] | None = None) -> CheckpointSaveResult:
        started = time.monotonic()
        state.assert_safe()
        if state.run_id != self.run_root.name:
            raise CheckpointError('State run_id does not match the managed run directory.')
        queue.validate()
        previous_index = state.checkpoint_index
        next_index = self._next_index(state)
        state.checkpoint_index = next_index
        state.updated_at = utc_now()
        temporary = self.checkpoint_root / f'.tmp-{uuid.uuid4().hex}'
        final = self.checkpoint_root / f'ckpt_{next_index:06d}'
        temporary.mkdir(parents=False, exist_ok=False)
        try:
            atomic_write_json(temporary / 'state.json', asdict(state))
            atomic_write_json(temporary / 'bounds.json', state.bounds)
            atomic_write_json(temporary / 'work_queue.json', queue.to_payload())
            atomic_write_json(temporary / 'meta.json', {
                'schema_version': 1,
                'saved_at': utc_now(),
                'config': self.config.canonical_payload(),
                'config_hash': self.config.config_hash(),
                'source_hash': self.source_hash,
                'notebook_hash': self.notebook_hash,
                'environment': environment_info(),
            })
            _write_pickle(temporary / 'rng_state.pkl', capture_rng_state())
            self.tensor_store.save(temporary / 'tensors', tensors or {})
            hashes: dict[str, str] = {}
            for path in sorted(temporary.rglob('*')):
                if path.is_file() and path.name not in {'hashes.json', 'COMMITTED'}:
                    hashes[path.relative_to(temporary).as_posix()] = sha256_file(path)
            atomic_write_json(temporary / 'hashes.json', hashes)
            fsync_directory(temporary)
            os.replace(temporary, final)
            fsync_directory(self.checkpoint_root)
            atomic_write_text(final / 'COMMITTED', utc_now())
            atomic_write_json(self.checkpoint_root / 'LATEST.json', {'checkpoint': final.name, 'saved_at': utc_now()})
            save_elapsed = time.monotonic() - started
            verify_started = time.monotonic()
            self.verify(final)
            verify_elapsed = time.monotonic() - verify_started
            return CheckpointSaveResult(final, next_index, directory_size(final), save_elapsed, verify_elapsed)
        except Exception:
            state.checkpoint_index = previous_index
            if temporary.exists():
                shutil.rmtree(temporary, ignore_errors=True)
            raise

    def verify(self, path: Path) -> None:
        if path.is_symlink() or path.parent.resolve() != self.checkpoint_root.resolve():
            raise CheckpointError(f'Checkpoint path is outside the managed root: {path}')
        if not path.is_dir() or not (path / 'COMMITTED').is_file():
            raise CheckpointError(f'Uncommitted checkpoint: {path}')
        if any(item.is_symlink() for item in path.rglob('*')):
            raise CheckpointError(f'Symlink found inside checkpoint: {path}')
        hashes_path = path / 'hashes.json'
        if not hashes_path.is_file():
            raise CheckpointError(f'Missing hashes.json: {path}')
        expected_payload = read_json(hashes_path)
        if not isinstance(expected_payload, dict) or any(
            not isinstance(relative, str) or not isinstance(digest, str)
            or len(digest) != 64 or any(character not in '0123456789abcdef' for character in digest)
            for relative, digest in expected_payload.items()
        ):
            raise CheckpointError(f'Invalid hash manifest structure: {path}')
        expected: dict[str, str] = expected_payload
        actual_files = {
            item.relative_to(path).as_posix()
            for item in path.rglob('*')
            if item.is_file() and item.name not in {'hashes.json', 'COMMITTED'}
        }
        if actual_files != set(expected):
            raise CheckpointError(f'Checkpoint file-set mismatch: {path}')
        for relative, digest in expected.items():
            if sha256_file(path / relative) != digest:
                raise CheckpointError(f'Checkpoint hash mismatch: {path / relative}')
        meta = read_json(path / 'meta.json')
        if not isinstance(meta, dict):
            raise CheckpointError(f'Invalid checkpoint metadata structure: {path}')
        if meta.get('config_hash') != self.config.config_hash():
            raise ConfigMismatchError(f'Checkpoint config mismatch: {path}')
        if meta.get('config') != self.config.canonical_payload():
            raise ConfigMismatchError(f'Checkpoint canonical config mismatch: {path}')
        if meta.get('source_hash') != self.source_hash:
            raise ConfigMismatchError(f'Checkpoint source hash mismatch: {path}')

    def load_latest(self, restore_rng: bool = True) -> LoadedCheckpoint | None:
        skipped: list[str] = []
        for path in self._candidate_paths():
            try:
                self.verify(path)
                state = RunState(**read_json(path / 'state.json'))
                state.assert_safe()
                if state.run_id != self.run_root.name:
                    raise CheckpointError(f'Checkpoint run_id/path mismatch: {path}')
                if state.config_hash != self.config.config_hash():
                    raise ConfigMismatchError(f'State config mismatch: {path}')
                if read_json(path / 'bounds.json') != state.bounds:
                    raise CheckpointError(f'Bounds/state mismatch: {path}')
                queue = WorkQueue.from_payload(read_json(path / 'work_queue.json'))
                try:
                    tensors = self.tensor_store.load(path / 'tensors')
                except Exception as exc:
                    raise CheckpointError(f'Tensor shard validation failed: {path}') from exc
                if restore_rng:
                    try:
                        with (path / 'rng_state.pkl').open('rb') as handle:
                            restore_rng_state(pickle.load(handle))
                    except Exception as exc:
                        raise CheckpointError(f'RNG restoration failed: {path}') from exc
                return LoadedCheckpoint(state, queue, path, tensors, tuple(skipped))
            except ConfigMismatchError:
                raise
            except (AttributeError, CheckpointError, EOFError, OSError, TypeError, ValueError, KeyError, pickle.PickleError) as exc:
                skipped.append(f'{path.name}: {exc}')
        return None

    def load_tensors(self, checkpoint_path: Path) -> dict[str, Any]:
        self.verify(checkpoint_path)
        return self.tensor_store.load(checkpoint_path / 'tensors')


In [ ]:
%%writefile src/reporting.py
from __future__ import annotations

import json
import resource
import shlex
from pathlib import Path
from typing import Any

from .checkpoint import CheckpointSaveResult, RunState
from .common import atomic_write_json, atomic_write_text, fsync_descriptor, fsync_directory, utc_now
from .runtime import PERSIST_ACK_ENV, PERSIST_ACK_TOKEN, PERSIST_ROOT_ENV
from .work_queue import WorkQueue

try:
    import torch
except ImportError:
    torch = None


class JsonlLogger:
    def __init__(self, path: Path) -> None:
        self.path = path
        path.parent.mkdir(parents=True, exist_ok=True)

    def emit(self, event: str, **payload: Any) -> None:
        record = {'timestamp': utc_now(), 'event': event, **payload}
        line = json.dumps(record, ensure_ascii=False, sort_keys=True, allow_nan=False) + '\n'
        with self.path.open('a', encoding='utf-8') as handle:
            handle.write(line)
            handle.flush()
            fsync_descriptor(handle.fileno())
        fsync_directory(self.path.parent)


def peak_memory_report() -> dict[str, int | None]:
    cpu_raw = int(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss)
    gpu_peak = None
    if torch is not None and torch.cuda.is_available():
        gpu_peak = int(torch.cuda.max_memory_allocated())
    return {'cpu_ru_maxrss_raw': cpu_raw, 'gpu_peak_allocated_bytes': gpu_peak}


def next_session_instructions(persistent_root: Path, run_id: str, project_root: Path) -> str:
    return '\n'.join((
        'NEXT SESSION (fresh kernel):',
        f'1. Mount the same durable storage containing {persistent_root}.',
        f'2. Set VALIDATED_RG_PROJECT_ROOT={shlex.quote(str(project_root))}.',
        f'3. Set {PERSIST_ROOT_ENV}={shlex.quote(str(persistent_root))}.',
        f'4. Set {PERSIST_ACK_ENV}={PERSIST_ACK_TOKEN}.',
        f'5. Set VALIDATED_RG_RUN_ID={run_id}.',
        '6. Rerun this notebook from the first cell in a fresh kernel.',
        '7. Confirm the M0 tests pass, then call orchestrator.run_until_checkpoint().',
    ))


def write_session_artifacts(
    run_root: Path,
    state: RunState,
    queue: WorkQueue,
    *,
    stop_reason: str,
    elapsed_s: float,
    remaining_s: float,
    persistent_root: Path,
    project_root: Path,
) -> dict[str, str]:
    state.assert_m0_safe()
    queue.validate()
    reports = run_root / 'reports'
    unfinished = [
        {
            'item_id': item.item_id, 'phase': item.phase, 'status': item.status,
            'attempts': item.attempts, 'last_error': item.last_error,
        }
        for item in queue.items.values() if item.status != 'done'
    ]
    counts = {
        status: sum(item.status == status for item in queue.items.values())
        for status in ('pending', 'running', 'done', 'failed', 'blocked_resource')
    }
    summary_path = reports / 'session_summary.json'
    metrics_path = reports / 'latest_metrics.json'
    next_path = reports / 'next_session_plan.md'
    atomic_write_json(summary_path, {
        'schema_version': 1, 'generated_at': utc_now(), 'run_id': state.run_id,
        'phase': state.phase, 'checkpoint_index': state.checkpoint_index,
        'stop_reason': stop_reason, 'elapsed_s': elapsed_s, 'remaining_s': remaining_s,
        'certification_status': state.certification_status, 'queue_counts': counts,
        'unfinished_work_items': unfinished,
        'mathematical_status': 'M0_ONLY_NOT_CERTIFIED',
        'unproved_or_unimplemented_scope': ['M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'P1-P13'],
    })
    atomic_write_json(metrics_path, {
        'generated_at': utc_now(), 'elapsed_s': elapsed_s, 'remaining_s': remaining_s,
        'phase': state.phase, 'checkpoint_index': state.checkpoint_index,
        'queue_counts': counts, 'memory': peak_memory_report(),
        'rigorous_error_bounds': {}, 'approximate_spectral_radius': None,
        'certification_status': state.certification_status,
    })
    if state.phase == 'M0_COMPLETE':
        plan = (
            '# Next action\n\nM0 completed. Review `M0_report.json` and its tests. '
            'Do not start M1 until the M0 acceptance result is confirmed in the execution environment.\n'
        )
    else:
        plan = '# Next session plan\n\n```text\n' + next_session_instructions(
            persistent_root, state.run_id, project_root
        ) + '\n```\n'
    atomic_write_text(next_path, plan)
    return {
        'session_summary': str(summary_path), 'latest_metrics': str(metrics_path),
        'next_session_plan': str(next_path),
    }


def write_m0_report(
    run_root: Path,
    state: RunState,
    queue: WorkQueue,
    test_report: dict[str, Any],
    checkpoint: CheckpointSaveResult | None,
    generated_files: list[str],
) -> Path:
    state.assert_m0_safe()
    report = {
        'milestone': 'M0',
        'generated_at': utc_now(),
        'run_id': state.run_id,
        'phase': state.phase,
        'certification_status': state.certification_status,
        'files_changed': generated_files,
        'tests': test_report,
        'restart_test_status': test_report.get('fresh_process_resume', 'UNKNOWN'),
        'remaining_todos': ['M1 exact 2D benchmark', 'M2 low-cutoff armillary', 'M3 GPU Triad-ATRG', 'M4 forward AD', 'M5 one-step validation', 'M6 multi-step certificate'],
        'heuristic_bounds': ['All RG, residual, tail, influence, and certificate bounds are absent in M0; none is treated as zero.'],
        'memory': peak_memory_report(),
        'checkpoint': None if checkpoint is None else {
            'path': str(checkpoint.path),
            'size_bytes': checkpoint.size_bytes,
            'save_s': checkpoint.save_s,
            'verify_s': checkpoint.verify_s,
        },
        'queue': {
            'done': sum(item.status == 'done' for item in queue.items.values()),
            'pending': sum(item.status == 'pending' for item in queue.items.values()),
            'running': sum(item.status == 'running' for item in queue.items.values()),
            'failed': sum(item.status == 'failed' for item in queue.items.values()),
        },
    }
    path = run_root / 'reports' / 'M0_report.json'
    atomic_write_json(path, report)
    return path


In [ ]:
%%writefile src/orchestrator.py
from __future__ import annotations

import json
import os
import random
import shutil
import time
import uuid
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np

from .checkpoint import CheckpointManager, CheckpointSaveResult, RunState
from .common import (
    atomic_write_json, fsync_descriptor, fsync_directory, fsync_file, hash_tree, read_json,
    sha256_file, utc_now,
)
from .config import RunConfig
from .reporting import (
    JsonlLogger, next_session_instructions, write_m0_report, write_session_artifacts,
)
from .runtime import environment_info, runtime_compatibility_signature
from .session_guard import SessionGuard, SessionState
from .work_queue import WorkItem, WorkQueue

try:
    import torch
except ImportError:
    torch = None

GENERATED_FILES = [
    'notebooks/20_m1_exact_2d.ipynb', 'README.md', 'requirements/paperspace.txt',
    'src/__init__.py', 'src/config.py', 'src/common.py', 'src/runtime.py',
    'src/session_guard.py', 'src/work_queue.py', 'src/checkpoint.py',
    'src/reporting.py', 'src/orchestrator.py', 'tests/test_m0.py', 'pytest.ini',
]

GOVERNING_DOCUMENTS = [
    'validated_4d_su2_rg_codex_design.md', 'AGENTS.md',
    'validated_4d_su2_rg_full_plan_bundle/validated_4d_su2_rg_codex_design_v0_2.md',
    'validated_4d_su2_rg_full_plan_bundle/M1_M6_VALIDATED_RG_ROADMAP.md',
    'validated_4d_su2_rg_full_plan_bundle/MATHEMATICAL_CERTIFICATION_SPEC.md',
    'validated_4d_su2_rg_full_plan_bundle/CODEX_PROMPTS_M1_M6.md',
    'validated_4d_su2_rg_full_plan_bundle/AGENTS_validated_4d_su2_rg_v0_2.md',
]

REFERENCE_ARTIFACTS = [
    'validated_4d_su2_rg_full_plan_bundle/validated_su2_rg_prototype.py',
    'validated_4d_su2_rg_full_plan_bundle/validated_su2_rg_report.md',
    'validated_4d_su2_rg_full_plan_bundle/validated_4d_su2_rg_M1_M6_tracker.ipynb',
]


def governing_document_hashes(project_root: Path) -> dict[str, str]:
    hashes: dict[str, str] = {}
    for relative in GOVERNING_DOCUMENTS:
        path = project_root / relative
        if not path.is_file():
            raise RunCompatibilityError(f'Missing governing document: {relative}')
        hashes[relative] = sha256_file(path)
    return hashes


def reference_artifact_hashes(project_root: Path) -> dict[str, str]:
    hashes: dict[str, str] = {}
    for relative in REFERENCE_ARTIFACTS:
        path = project_root / relative
        if not path.is_file():
            raise RunCompatibilityError(f'Missing full-plan reference artifact: {relative}')
        hashes[relative] = sha256_file(path)
    return hashes


class RunCompatibilityError(RuntimeError):
    '''Raised when an existing durable run cannot be resumed exactly.'''


class Orchestrator:
    def __init__(
        self,
        persistent_root: Path,
        run_root: Path,
        project_root: Path,
        config: RunConfig,
        state: RunState,
        queue: WorkQueue,
        checkpoints: CheckpointManager,
        prefer_cuda: bool,
        test_report: dict[str, Any],
    ) -> None:
        self.persistent_root = persistent_root
        self.run_root = run_root
        self.project_root = project_root
        self.config = config
        self.state = state
        self.queue = queue
        self.checkpoints = checkpoints
        self.prefer_cuda = prefer_cuda
        self.test_report = test_report
        self.guard = SessionGuard(config)
        self.logger = JsonlLogger(run_root / 'logs' / 'events.jsonl')
        self.tensors: dict[str, Any] = {}
        self.last_checkpoint: CheckpointSaveResult | None = None

    def checkpoint(self, reason: str) -> CheckpointSaveResult:
        self.state.assert_m0_safe()
        self.state.notes.append(f'{utc_now()} checkpoint: {reason}')
        result = self.checkpoints.save(self.state, self.queue, self.tensors)
        self.guard.mark_checkpoint()
        self.last_checkpoint = result
        self.logger.emit(
            'checkpoint_committed', run_id=self.state.run_id, phase=self.state.phase,
            checkpoint=result.index, reason=reason, size_bytes=result.size_bytes,
            save_s=result.save_s, verify_s=result.verify_s,
        )
        print(f'Checkpoint {result.index:06d} committed and verified: {result.path}')
        return result

    def _execute_dummy(self, item: WorkItem) -> tuple[str, str]:
        size = int(item.parameters.get('size', self.config.dummy_size))
        steps = int(item.parameters.get('steps', self.config.dummy_steps))
        if size <= 0 or steps <= 0:
            raise ValueError('Dummy dimensions and steps must be positive.')
        artifact_parent = self.run_root / 'artifacts' / item.item_id
        artifact_parent.mkdir(parents=True, exist_ok=True)
        temporary = artifact_parent / f'.tmp-attempt-{item.attempts:03d}-{uuid.uuid4().hex}'
        final = artifact_parent / f'attempt_{item.attempts:03d}'
        if final.exists():
            raise RuntimeError(f'Attempt output already exists: {final}')
        temporary.mkdir(parents=False, exist_ok=False)
        try:
            if torch is not None:
                device = 'cuda' if self.prefer_cuda and torch.cuda.is_available() else 'cpu'
                value = torch.eye(size, dtype=torch.float64, device=device)
                for _ in range(steps):
                    value = value @ value.mT
                    norm = torch.linalg.vector_norm(value)
                    if not bool(torch.isfinite(norm)) or float(norm) <= 0.0:
                        raise FloatingPointError('Dummy torch workload produced invalid normalization.')
                    value = value / norm
                if not bool(torch.isfinite(value).all()):
                    raise FloatingPointError('Dummy torch workload produced NaN or Inf.')
                result_file = temporary / 'result.pt'
                torch.save(value[:8, :8].cpu(), result_file)
                fsync_file(result_file)
                device_name = device
            else:
                value = np.eye(size, dtype=np.float64)
                for _ in range(steps):
                    value = value @ value.T
                    norm = np.linalg.norm(value)
                    if not np.isfinite(norm) or norm <= 0.0:
                        raise FloatingPointError('Dummy NumPy workload produced invalid normalization.')
                    value = value / norm
                if not np.isfinite(value).all():
                    raise FloatingPointError('Dummy NumPy workload produced NaN or Inf.')
                result_file = temporary / 'result.npy'
                with result_file.open('wb') as handle:
                    np.save(handle, value[:8, :8], allow_pickle=False)
                    handle.flush()
                    fsync_descriptor(handle.fileno())
                device_name = 'cpu-numpy'
            atomic_write_json(temporary / 'metadata.json', {
                'item_id': item.item_id, 'attempt': item.attempts, 'device': device_name,
                'certification_status': 'NOT_CERTIFIED', 'created_at': utc_now(),
            })
            fsync_directory(temporary)
            os.replace(temporary, final)
            fsync_directory(artifact_parent)
            committed_result = final / result_file.name
            relative = committed_result.relative_to(self.run_root).as_posix()
            digest = sha256_file(committed_result)
            marker = self.run_root / 'work_items' / f'{item.item_id}.done'
            atomic_write_json(marker, {
                'item_id': item.item_id, 'result_relpath': relative,
                'result_sha256': digest, 'completed_at': utc_now(),
            })
            if sha256_file(committed_result) != digest:
                raise RuntimeError('Result verification failed immediately after commit.')
            return relative, digest
        except Exception:
            if temporary.exists():
                shutil.rmtree(temporary, ignore_errors=True)
            raise

    def _summary(self, stop_reason: str) -> dict[str, Any]:
        self.state.assert_m0_safe()
        elapsed_s = self.guard.elapsed_s()
        remaining_s = self.guard.remaining_s()
        summary = {
            'run_id': self.state.run_id,
            'phase': self.state.phase,
            'checkpoint_index': self.state.checkpoint_index,
            'certification_status': self.state.certification_status,
            'elapsed_s': elapsed_s,
            'remaining_s': remaining_s,
            'session_state': self.guard.state().value,
            'stop_reason': stop_reason,
        }
        summary['session_artifacts'] = write_session_artifacts(
            self.run_root, self.state, self.queue, stop_reason=stop_reason,
            elapsed_s=elapsed_s, remaining_s=remaining_s,
            persistent_root=self.persistent_root, project_root=self.project_root,
        )
        print(json.dumps(summary, ensure_ascii=False, indent=2, allow_nan=False))
        if self.state.phase == 'M0_COMPLETE':
            print('NEXT ACTION: inspect reports/M0_report.json; M0 is complete and M1 is not started automatically.')
        else:
            print(next_session_instructions(self.persistent_root, self.state.run_id, self.project_root))
        return summary

    def run_until_checkpoint(self) -> dict[str, Any]:
        self.state.assert_m0_safe()
        self.logger.emit('session_started', run_id=self.state.run_id, phase=self.state.phase)
        stop_reason = 'unknown'
        while True:
            session_state = self.guard.state()
            if session_state is SessionState.RETURN:
                stop_reason = 'hard return threshold reached; returning with last verified checkpoint'
                break
            if session_state in {SessionState.DRAIN, SessionState.FINAL_SAVE}:
                self.checkpoint(f'session state {session_state.value}')
                stop_reason = f'{session_state.value.lower()} checkpoint complete'
                break
            if self.guard.checkpoint_due():
                self.checkpoint('periodic 15-minute checkpoint')
            item = self.queue.next_pending()
            if item is None:
                incomplete = [
                    queued for queued in self.queue.items.values() if queued.status != 'done'
                ]
                if incomplete:
                    self.checkpoint('queue has no runnable item but contains incomplete work')
                    summary = ', '.join(f'{queued.item_id[:12]}:{queued.status}' for queued in incomplete)
                    raise RuntimeError(f'M0 cannot complete with incomplete work items: {summary}')
                self.state.phase = 'M0_COMPLETE'
                self.checkpoint('M0 work queue complete')
                report = write_m0_report(
                    self.run_root, self.state, self.queue, self.test_report,
                    self.last_checkpoint, GENERATED_FILES,
                )
                self.logger.emit('milestone_complete', run_id=self.state.run_id, phase=self.state.phase, report=str(report))
                stop_reason = f'M0 complete; report written to {report}'
                break
            predicted_s = self.queue.predicted_duration(item)
            if not self.guard.may_start(predicted_s):
                self.checkpoint('insufficient safe time for next bounded item')
                stop_reason = 'next work item deferred to a fresh session'
                break
            item.attempts += 1
            if item.attempts > self.config.max_item_attempts:
                item.status = 'failed'
                item.last_error = 'Maximum attempt count exceeded.'
                self.checkpoint('item exceeded maximum attempts')
                raise RuntimeError(f'Work item permanently failed: {item.item_id}')
            item.status = 'running'
            self.state.phase = item.phase
            self.checkpoint(f'before work item {item.item_id[:12]}')
            started = time.monotonic()
            try:
                if item.phase != 'DUMMY':
                    raise RunCompatibilityError(f'Queue phase {item.phase!r} is outside the approved M0 scope.')
                relative, digest = self._execute_dummy(item)
                elapsed = time.monotonic() - started
                item.result_relpath = relative
                item.result_sha256 = digest
                item.status = 'done'
                item.last_error = None
                self.queue.record_timing(item.phase, elapsed)
                self.logger.emit(
                    'item_completed', run_id=self.state.run_id, phase=item.phase,
                    item_id=item.item_id, elapsed_s=elapsed, result_sha256=digest,
                )
                if self.guard.state() is SessionState.RETURN:
                    stop_reason = (
                        'hard return reached after artifact commit; the prior running-item checkpoint '
                        'will repair from the verified done marker on resume'
                    )
                    break
                self.checkpoint(f'after work item {item.item_id[:12]}')
            except KeyboardInterrupt:
                item.status = 'pending'
                item.last_error = 'KeyboardInterrupt'
                item.result_relpath = None
                item.result_sha256 = None
                if self.guard.state() is not SessionState.RETURN:
                    self.checkpoint('KeyboardInterrupt recovery')
                print(next_session_instructions(self.persistent_root, self.state.run_id, self.project_root))
                raise
            except Exception as exc:
                item.last_error = repr(exc)
                item.status = 'failed' if item.attempts >= self.config.max_item_attempts else 'pending'
                item.result_relpath = None
                item.result_sha256 = None
                if self.guard.state() is not SessionState.RETURN:
                    self.checkpoint('work item exception')
                print(next_session_instructions(self.persistent_root, self.state.run_id, self.project_root))
                raise
        self.logger.emit('session_stopped', run_id=self.state.run_id, reason=stop_reason)
        return self._summary(stop_reason)


def _notebook_hash(project_root: Path) -> str | None:
    notebook = project_root / 'notebooks/20_m1_exact_2d.ipynb'
    return sha256_file(notebook) if notebook.is_file() else None


def _seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    if torch is not None:
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
            torch.cuda.reset_peak_memory_stats()


def create_or_resume(
    persistent_root: Path,
    config: RunConfig,
    project_root: Path,
    run_id: str | None = None,
    prefer_cuda: bool = True,
    test_report: dict[str, Any] | None = None,
) -> Orchestrator:
    if prefer_cuda != config.prefer_cuda:
        raise RunCompatibilityError('prefer_cuda differs from the immutable run configuration.')
    config_hash = config.config_hash()
    source_hash = hash_tree(project_root / 'src')
    notebook_hash = _notebook_hash(project_root)
    document_hashes = governing_document_hashes(project_root)
    reference_hashes = reference_artifact_hashes(project_root)
    current_environment = environment_info()
    current_runtime_signature = runtime_compatibility_signature(current_environment)
    runs_root = persistent_root / 'runs'
    runs_root.mkdir(parents=True, exist_ok=True)
    requested = run_id or os.environ.get('VALIDATED_RG_RUN_ID')
    latest_pointer = persistent_root / 'LATEST_RUN.json'
    if requested is None and latest_pointer.is_file():
        pointer = read_json(latest_pointer)
        if not isinstance(pointer, dict):
            raise RunCompatibilityError('LATEST_RUN.json is malformed; choose a run ID explicitly.')
        if pointer.get('config_hash') == config_hash:
            requested = pointer.get('run_id')
            if not isinstance(requested, str):
                raise RunCompatibilityError('LATEST_RUN.json has no valid run_id; choose one explicitly.')
    if requested is None:
        requested = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + config_hash[:12]
    if not isinstance(requested, str) or not requested or '/' in requested or '\\' in requested or requested in {'.', '..'}:
        raise ValueError(f'Unsafe run_id: {requested!r}')
    run_root = runs_root / requested
    if run_root.exists() and (run_root / 'run_config.json').is_file():
        stored_config = read_json(run_root / 'run_config.json')
        if stored_config != config.canonical_payload():
            raise RunCompatibilityError('Immutable run configuration differs from the stored run.')
        manifest_path = run_root / 'run_manifest.json'
        if not manifest_path.is_file():
            raise RunCompatibilityError('Run directory is incomplete (run_manifest.json is missing).')
        manifest = read_json(manifest_path)
        if not isinstance(manifest, dict) or (
            manifest.get('run_id') != requested
            or manifest.get('config_hash') != config_hash
            or manifest.get('certification_status') != 'NOT_CERTIFIED'
            or manifest.get('sector_ordering') != 'M0_NONE'
            or manifest.get('normalization_convention') != 'M0_NONE'
            or manifest.get('wigner_convention') != 'M0_NONE'
        ):
            raise RunCompatibilityError('Run manifest identity or certification invariant is invalid.')
        if manifest.get('source_hash') != source_hash:
            raise RunCompatibilityError('Generated source changed; start a new run or perform an explicit migration.')
        if manifest.get('governing_document_hashes') != document_hashes:
            raise RunCompatibilityError('A governing M0/M1-M6 document changed; start a new run or audit explicitly.')
        if manifest.get('reference_artifact_hashes') != reference_hashes:
            raise RunCompatibilityError('A full-plan prototype/report/tracker changed; start a new run or audit explicitly.')
        if manifest.get('runtime_compatibility') != current_runtime_signature:
            raise RunCompatibilityError(
                'Python/NumPy/PyTorch/CUDA/GPU runtime changed; resume with the original Paperspace runtime.'
            )
        required_directories = ('logs', 'reports', 'artifacts', 'work_items', 'checkpoints')
        missing_directories = [name for name in required_directories if not (run_root / name).is_dir()]
        if missing_directories:
            raise RunCompatibilityError(f'Run directory is incomplete; missing directories: {missing_directories}')
        manager = CheckpointManager(run_root, config, source_hash, notebook_hash)
        loaded = manager.load_latest(restore_rng=True)
        if loaded is None:
            raise RunCompatibilityError('Existing run has no valid checkpoint.')
        repaired = loaded.queue.recover_interrupted(run_root)
        orchestrator = Orchestrator(
            persistent_root, run_root, project_root, config, loaded.state, loaded.queue,
            manager, prefer_cuda, test_report or {},
        )
        orchestrator.tensors = loaded.tensors
        print('Resumed from:', loaded.path)
        for warning in loaded.skipped_invalid:
            print('Fallback skipped invalid checkpoint:', warning)
        if repaired:
            orchestrator.checkpoint(f'recovered {len(repaired)} interrupted work item(s)')
        return orchestrator
    if run_root.exists():
        raise RunCompatibilityError(
            'Run directory exists but is incomplete (run_config.json is missing); refusing to overwrite it.'
        )
    run_root.mkdir(parents=True, exist_ok=False)
    for relative in ('logs', 'reports', 'artifacts', 'work_items', 'checkpoints'):
        (run_root / relative).mkdir(parents=True, exist_ok=True)
    manager = CheckpointManager(run_root, config, source_hash, notebook_hash)
    atomic_write_json(run_root / 'run_config.json', config.canonical_payload())
    atomic_write_json(run_root / 'run_manifest.json', {
        'schema_version': 1, 'run_id': requested, 'created_at': utc_now(),
        'config_hash': config_hash, 'source_hash': source_hash,
        'notebook_hash': notebook_hash, 'environment': current_environment,
        'runtime_compatibility': current_runtime_signature,
        'governing_document_hashes': document_hashes,
        'reference_artifact_hashes': reference_hashes,
        'sector_ordering': 'M0_NONE', 'normalization_convention': 'M0_NONE',
        'wigner_convention': 'M0_NONE', 'certification_status': 'NOT_CERTIFIED',
    })
    _seed_everything(config.seed)
    state = RunState(requested, config_hash, utc_now(), utc_now())
    queue = WorkQueue()
    input_hash = config_hash
    for index in range(config.dummy_items):
        queue.add(
            'DUMMY', input_hash, {'index': index, 'size': config.dummy_size, 'steps': config.dummy_steps},
            predicted_s=config.dummy_predicted_s,
        )
    orchestrator = Orchestrator(
        persistent_root, run_root, project_root, config, state, queue, manager,
        prefer_cuda, test_report or {},
    )
    orchestrator.checkpoint('initial run state')
    atomic_write_json(latest_pointer, {'run_id': requested, 'config_hash': config_hash, 'updated_at': utc_now()})
    return orchestrator


In [ ]:
%%writefile pytest.ini
[pytest]
markers =
    gpu: optional CUDA-only test
testpaths = tests
addopts = --strict-markers


In [ ]:
%%writefile tests/test_m0.py
from __future__ import annotations

import json
import os
import random
import subprocess
import sys
from pathlib import Path

import numpy as np
import pytest

from src.checkpoint import CheckpointManager, RunState
from src.common import atomic_write_json, hash_tree, sha256_file, utc_now
from src.config import ConfigError, RunConfig
from src.orchestrator import RunCompatibilityError, create_or_resume
from src.runtime import PERSIST_ACK_TOKEN, PersistentRootError, validate_persistent_root
from src.session_guard import SessionGuard, SessionState
from src.work_queue import WorkItem, WorkQueue

try:
    import torch
except ImportError:
    torch = None

PROJECT_ROOT = Path(__file__).resolve().parents[1]
SOURCE_HASH = hash_tree(PROJECT_ROOT / 'src')


class FakeClock:
    def __init__(self, value: float = 0.0) -> None:
        self.value = value

    def __call__(self) -> float:
        return self.value


def manager_for(run_root: Path, config: RunConfig) -> CheckpointManager:
    run_root.mkdir(parents=True, exist_ok=True)
    return CheckpointManager(run_root, config, SOURCE_HASH, None)


def new_state(config: RunConfig, run_id: str = 'run') -> RunState:
    return RunState(run_id, config.config_hash(), utc_now(), utc_now())


def test_config_is_immutable_and_fail_closed() -> None:
    config = RunConfig()
    assert config.certification_status == 'NOT_CERTIFIED'
    with pytest.raises(ConfigError):
        RunConfig(certification_status='CERTIFIED')
    with pytest.raises(ConfigError):
        RunConfig(hard_return_s=5.5 * 3600 + 1)
    with pytest.raises(ConfigError):
        RunConfig(dummy_size=0)
    with pytest.raises(ConfigError):
        RunConfig(dummy_predicted_s=21 * 60.0)
    with pytest.raises(ConfigError):
        RunConfig(prefer_cuda='yes')
    with pytest.raises(ConfigError):
        RunConfig(dummy_items=1.5)


def test_persistent_root_is_explicit_and_rejects_ephemeral(tmp_path: Path) -> None:
    with pytest.raises(PersistentRootError):
        validate_persistent_root(str(tmp_path), acknowledgement=None)
    with pytest.raises(PersistentRootError):
        validate_persistent_root('/tmp/validated-rg', acknowledgement=PERSIST_ACK_TOKEN)
    with pytest.raises(PersistentRootError):
        validate_persistent_root('/private/var/folders/validated-rg', acknowledgement=PERSIST_ACK_TOKEN)


def test_session_guard_boundaries_and_prediction() -> None:
    config = RunConfig(
        checkpoint_interval_s=1.0, max_work_item_s=2.0, short_task_limit_s=0.5,
        checkpoint_reserve_s=0.1, no_long_task_after_s=3.0, drain_after_s=4.0,
        final_save_after_s=5.0, hard_return_s=6.0, dummy_predicted_s=1.0,
    )
    clock = FakeClock()
    guard = SessionGuard(config, clock=clock)
    assert guard.state() is SessionState.RUN
    assert guard.may_start(1.0)
    assert not guard.checkpoint_due()
    clock.value = 1.0
    assert guard.checkpoint_due()
    guard.mark_checkpoint()
    assert not guard.checkpoint_due()
    clock.value = 3.0
    assert guard.state() is SessionState.NO_LONG_TASK
    assert not guard.may_start(1.0)
    clock.value = 4.0
    assert guard.state() is SessionState.DRAIN
    clock.value = 5.0
    assert guard.state() is SessionState.FINAL_SAVE
    clock.value = 6.0
    assert guard.state() is SessionState.RETURN


def test_work_queue_rejects_content_hash_drift() -> None:
    queue = WorkQueue()
    item_id = queue.add('DUMMY', 'input', {'index': 0}, 1.0)
    payload = queue.to_payload()
    payload['items'][item_id]['parameters']['index'] = 1
    with pytest.raises(ValueError, match='content hash mismatch'):
        WorkQueue.from_payload(payload)


def test_atomic_checkpoint_and_sharded_round_trip(tmp_path: Path) -> None:
    config = RunConfig(tensor_shard_bytes=64)
    manager = manager_for(tmp_path / 'run', config)
    state = new_state(config)
    queue = WorkQueue()
    queue.add('DUMMY', config.config_hash(), {'index': 0}, 1.0)
    array = np.arange(256, dtype=np.float64).reshape(32, 8)
    empty = np.empty((0, 8), dtype=np.float64)
    scalar = np.asarray(7.0, dtype=np.float64)
    saved = manager.save(state, queue, {'array': array, 'empty': empty, 'scalar': scalar})
    assert (saved.path / 'COMMITTED').is_file()
    manager.verify(saved.path)
    index = json.loads((saved.path / 'tensors' / 'index.json').read_text())
    assert len(index['array']['files']) > 1
    for filename in index['array']['files']:
        assert np.load(saved.path / 'tensors' / filename, allow_pickle=False).nbytes <= 64
    loaded = manager.load_latest(restore_rng=False)
    assert loaded is not None and loaded.path == saved.path
    np.testing.assert_array_equal(loaded.tensors['array'], array)
    restored = manager.load_tensors(saved.path)
    np.testing.assert_array_equal(restored['array'], array)
    np.testing.assert_array_equal(restored['empty'], empty)
    np.testing.assert_array_equal(restored['scalar'], scalar)


def test_corrupt_latest_falls_back(tmp_path: Path) -> None:
    config = RunConfig()
    manager = manager_for(tmp_path / 'run', config)
    state = new_state(config)
    queue = WorkQueue()
    first = manager.save(state, queue)
    state.notes.append('second')
    second = manager.save(state, queue)
    (second.path / 'state.json').write_text('corrupt', encoding='utf-8')
    loaded = manager.load_latest(restore_rng=False)
    assert loaded is not None
    assert loaded.path == first.path
    assert loaded.skipped_invalid and second.path.name in loaded.skipped_invalid[0]


def test_rng_restoration(tmp_path: Path) -> None:
    config = RunConfig()
    manager = manager_for(tmp_path / 'run', config)
    random.seed(17)
    np.random.seed(19)
    if torch is not None:
        torch.manual_seed(23)
    state = new_state(config)
    saved = manager.save(state, WorkQueue())
    expected_python = random.random()
    expected_numpy = float(np.random.random())
    expected_torch = float(torch.rand(())) if torch is not None else None
    for _ in range(5):
        random.random(); np.random.random()
        if torch is not None:
            torch.rand(())
    loaded = manager.load_latest(restore_rng=True)
    assert loaded is not None and loaded.path == saved.path
    assert random.random() == expected_python
    assert float(np.random.random()) == expected_numpy
    if torch is not None:
        assert float(torch.rand(())) == expected_torch


def test_interrupted_item_recovery_requires_done_marker(tmp_path: Path) -> None:
    run_root = tmp_path / 'run'
    run_root.mkdir()
    queue = WorkQueue()
    item_id = queue.add('DUMMY', 'input', {'index': 0}, 1.0)
    item = queue.items[item_id]
    item.status = 'running'
    assert queue.recover_interrupted(run_root) == [item_id]
    assert item.status == 'pending'
    result = run_root / 'artifacts' / 'result.bin'
    result.parent.mkdir(parents=True)
    result.write_bytes(b'ok')
    digest = sha256_file(result)
    marker = run_root / 'work_items' / f'{item_id}.done'
    atomic_write_json(marker, {'item_id': item_id, 'result_relpath': 'artifacts/result.bin', 'result_sha256': digest})
    item.status = 'running'
    queue.recover_interrupted(run_root)
    assert item.status == 'done' and item.result_sha256 == digest
    result.unlink()
    queue.recover_interrupted(run_root)
    assert item.status == 'pending' and item.result_sha256 is None
    outside = tmp_path / 'outside.bin'
    outside.write_bytes(b'outside')
    atomic_write_json(marker, {
        'item_id': item_id, 'result_relpath': '../outside.bin',
        'result_sha256': sha256_file(outside),
    })
    item.status = 'running'
    queue.recover_interrupted(run_root)
    assert item.status == 'pending'


def test_resume_restores_checkpointed_tensor_state(tmp_path: Path) -> None:
    config = RunConfig(dummy_items=0, tensor_shard_bytes=64, prefer_cuda=False)
    first = create_or_resume(tmp_path, config, PROJECT_ROOT, run_id='tensor-resume', prefer_cuda=False)
    expected = np.arange(24, dtype=np.float64).reshape(3, 8)
    first.tensors['resume_array'] = expected
    first.checkpoint('tensor resume test')
    resumed = create_or_resume(tmp_path, config, PROJECT_ROOT, run_id='tensor-resume', prefer_cuda=False)
    np.testing.assert_array_equal(resumed.tensors['resume_array'], expected)


def test_incomplete_existing_run_fails_closed(tmp_path: Path) -> None:
    (tmp_path / 'runs' / 'partial-run').mkdir(parents=True)
    with pytest.raises(RunCompatibilityError, match='incomplete'):
        create_or_resume(tmp_path, RunConfig(prefer_cuda=False), PROJECT_ROOT, run_id='partial-run', prefer_cuda=False)


def test_resume_rejects_runtime_signature_drift(tmp_path: Path) -> None:
    config = RunConfig(dummy_items=0, prefer_cuda=False)
    created = create_or_resume(
        tmp_path, config, PROJECT_ROOT, run_id='runtime-drift', prefer_cuda=False,
    )
    manifest_path = created.run_root / 'run_manifest.json'
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    manifest['runtime_compatibility']['numpy'] = 'incompatible-test-version'
    atomic_write_json(manifest_path, manifest)
    with pytest.raises(RunCompatibilityError, match='runtime changed'):
        create_or_resume(
            tmp_path, config, PROJECT_ROOT, run_id='runtime-drift', prefer_cuda=False,
        )


def test_failed_item_cannot_be_reported_as_m0_complete(tmp_path: Path) -> None:
    config = RunConfig(dummy_items=1, dummy_size=8, dummy_steps=1, prefer_cuda=False)
    orchestrator = create_or_resume(tmp_path, config, PROJECT_ROOT, run_id='failed-run', prefer_cuda=False)
    item = next(iter(orchestrator.queue.items.values()))
    item.status = 'failed'
    item.last_error = 'synthetic failure'
    with pytest.raises(RuntimeError, match='cannot complete'):
        orchestrator.run_until_checkpoint()
    assert orchestrator.state.phase != 'M0_COMPLETE'


def test_cpu_only_dummy_orchestrator(tmp_path: Path) -> None:
    config = RunConfig(dummy_items=2, dummy_size=8, dummy_steps=1, tensor_shard_bytes=64, prefer_cuda=False)
    orchestrator = create_or_resume(tmp_path, config, PROJECT_ROOT, run_id='cpu-run', prefer_cuda=False, test_report={'fresh_process_resume': 'PASS'})
    summary = orchestrator.run_until_checkpoint()
    assert summary['phase'] == 'M0_COMPLETE'
    assert summary['certification_status'] == 'NOT_CERTIFIED'
    assert all(item.status == 'done' for item in orchestrator.queue.items.values())
    assert (orchestrator.run_root / 'reports' / 'M0_report.json').is_file()
    assert (orchestrator.run_root / 'reports' / 'session_summary.json').is_file()
    assert (orchestrator.run_root / 'reports' / 'latest_metrics.json').is_file()
    assert (orchestrator.run_root / 'reports' / 'next_session_plan.md').is_file()


def test_short_simulated_session_returns_after_drain_checkpoint(tmp_path: Path) -> None:
    config = RunConfig(
        dummy_items=1, dummy_size=8, dummy_steps=1, checkpoint_interval_s=1.0,
        max_work_item_s=2.0, short_task_limit_s=0.5, checkpoint_reserve_s=0.1,
        no_long_task_after_s=3.0, drain_after_s=4.0, final_save_after_s=5.0,
        hard_return_s=6.0, dummy_predicted_s=1.0, prefer_cuda=False,
    )
    orchestrator = create_or_resume(tmp_path, config, PROJECT_ROOT, run_id='timer-run', prefer_cuda=False)
    clock = FakeClock(0.0)
    orchestrator.guard = SessionGuard(config, clock=clock)
    clock.value = 4.0
    summary = orchestrator.run_until_checkpoint()
    assert summary['session_state'] == 'DRAIN'
    assert summary['certification_status'] == 'NOT_CERTIFIED'
    assert orchestrator.state.checkpoint_index >= 2


def test_hard_return_uses_done_marker_instead_of_late_checkpoint(tmp_path: Path) -> None:
    config = RunConfig(
        dummy_items=1, dummy_size=8, dummy_steps=1, dummy_predicted_s=0.1,
        checkpoint_interval_s=1.0, max_work_item_s=2.0, short_task_limit_s=0.5,
        checkpoint_reserve_s=0.1, no_long_task_after_s=3.0, drain_after_s=4.0,
        final_save_after_s=5.0, hard_return_s=6.0, prefer_cuda=False,
    )
    orchestrator = create_or_resume(tmp_path, config, PROJECT_ROOT, run_id='hard-return', prefer_cuda=False)
    clock = FakeClock(0.0)
    orchestrator.guard = SessionGuard(config, clock=clock)
    original_execute = orchestrator._execute_dummy
    def finish_at_deadline(item: WorkItem) -> tuple[str, str]:
        artifact = original_execute(item)
        clock.value = config.hard_return_s
        return artifact
    orchestrator._execute_dummy = finish_at_deadline
    summary = orchestrator.run_until_checkpoint()
    assert summary['session_state'] == 'RETURN'
    assert 'done marker' in summary['stop_reason']
    resumed = create_or_resume(tmp_path, config, PROJECT_ROOT, run_id='hard-return', prefer_cuda=False)
    assert all(item.status == 'done' for item in resumed.queue.items.values())


def test_fresh_process_resume(tmp_path: Path) -> None:
    config = RunConfig(dummy_items=2, dummy_size=8, dummy_steps=1, prefer_cuda=False)
    orchestrator = create_or_resume(tmp_path, config, PROJECT_ROOT, run_id='fresh-process', prefer_cuda=False)
    first = orchestrator.queue.next_pending()
    assert first is not None
    first.attempts += 1
    first.status = 'running'
    orchestrator.checkpoint('fresh-process test before first item')
    relative, digest = orchestrator._execute_dummy(first)
    first.result_relpath = relative
    first.result_sha256 = digest
    first.status = 'done'
    orchestrator.checkpoint('fresh-process test after first item')
    code = (
        'from pathlib import Path; '
        'from src.config import RunConfig; '
        'from src.orchestrator import create_or_resume; '
        f'c=RunConfig(dummy_items=2, dummy_size=8, dummy_steps=1, prefer_cuda=False); '
        f'o=create_or_resume(Path({str(tmp_path)!r}), c, Path({str(PROJECT_ROOT)!r}), run_id="fresh-process", prefer_cuda=False); '
        's=o.run_until_checkpoint(); assert s["phase"]=="M0_COMPLETE"; '
        'assert all(i.status=="done" for i in o.queue.items.values()); print(s["phase"])'
    )
    env = os.environ.copy()
    env['PYTHONPATH'] = str(PROJECT_ROOT) + os.pathsep + env.get('PYTHONPATH', '')
    completed = subprocess.run([sys.executable, '-c', code], cwd=PROJECT_ROOT, env=env, capture_output=True, text=True, timeout=60, check=False)
    assert completed.returncode == 0, completed.stderr
    assert 'M0_COMPLETE' in completed.stdout


@pytest.mark.gpu
def test_optional_cuda_dummy(tmp_path: Path) -> None:
    if torch is None or not torch.cuda.is_available():
        pytest.skip('CUDA is unavailable.')
    rng_config = RunConfig()
    rng_manager = manager_for(tmp_path / 'cuda-rng', rng_config)
    torch.cuda.manual_seed_all(29)
    rng_manager.save(new_state(rng_config, run_id='cuda-rng'), WorkQueue())
    expected_cuda = float(torch.rand((), device='cuda').cpu())
    for _ in range(5):
        torch.rand((), device='cuda')
    assert rng_manager.load_latest(restore_rng=True) is not None
    assert float(torch.rand((), device='cuda').cpu()) == expected_cuda
    config = RunConfig(dummy_items=1, dummy_size=8, dummy_steps=1)
    orchestrator = create_or_resume(tmp_path, config, PROJECT_ROOT, run_id='gpu-run', prefer_cuda=True)
    summary = orchestrator.run_until_checkpoint()
    assert summary['certification_status'] == 'NOT_CERTIFIED'
    assert torch.backends.cuda.matmul.allow_tf32 is False
    assert torch.backends.cudnn.allow_tf32 is False


In [ ]:
%%writefile tests/m1_helpers.py
from __future__ import annotations

from dataclasses import replace
from pathlib import Path
from typing import Any

from src.common import atomic_write_json, atomic_write_text, sha256_file
from src.m1_config import M1Config
from src.work_queue import WorkQueue


def make_synthetic_accepted_parent(tmp_path: Path, base: M1Config | None = None) -> M1Config:
    config = base or M1Config()
    parent_run_root = tmp_path / 'accepted-storage' / 'runs' / config.parent_run_id
    checkpoint = parent_run_root / 'checkpoints' / 'ckpt_000014'
    checkpoint.mkdir(parents=True, exist_ok=False)
    state = {
        'run_id': config.parent_run_id, 'config_hash': '0' * 64,
        'created_at': '2026-07-19T12:04:06+00:00', 'updated_at': '2026-07-19T12:04:08+00:00',
        'phase': 'M0_COMPLETE', 'checkpoint_index': 14,
        'certification_status': 'NOT_CERTIFIED',
    }
    queue = WorkQueue()
    for index in range(6):
        item_id = queue.add('DUMMY', 'synthetic-parent', {'index': index}, predicted_s=1.0)
        item = queue.items[item_id]; item.status = 'done'
        result = parent_run_root / 'artifacts' / str(index) / 'result.bin'
        result.parent.mkdir(parents=True, exist_ok=True); result.write_bytes(f'result-{index}'.encode('ascii'))
        item.result_relpath = result.relative_to(parent_run_root).as_posix()
        item.result_sha256 = sha256_file(result)
        marker = parent_run_root / 'work_items' / f'{item_id}.done'
        marker.parent.mkdir(parents=True, exist_ok=True)
        atomic_write_json(marker, {
            'item_id': item_id, 'result_relpath': item.result_relpath,
            'result_sha256': item.result_sha256,
        })
    atomic_write_json(checkpoint / 'state.json', state)
    atomic_write_json(checkpoint / 'work_queue.json', queue.to_payload())
    hashes = {name: sha256_file(checkpoint / name) for name in ('state.json', 'work_queue.json')}
    atomic_write_json(checkpoint / 'hashes.json', hashes)
    atomic_write_text(checkpoint / 'COMMITTED', 'committed\n')
    return replace(config, parent_checkpoint_path=str(checkpoint))


def passing_test_report() -> dict[str, Any]:
    return {
        'm0_regression_cpu_suite': 'PASS', 'm1_required_cpu_suite': 'PASS',
        'optional_gpu_suite': 'NOT_RUN_NO_CUDA', 'm1_fresh_process_resume': 'PASS',
    }


In [ ]:
%%writefile tests/test_m1_exact_arithmetic.py
from __future__ import annotations

from fractions import Fraction

import pytest

from src.exact_arithmetic import ExactArithmeticError, RationalInterval
from src.su2_representations import Irrep


def test_interval_exact_operations_and_round_trip() -> None:
    left = RationalInterval(Fraction(1, 3), Fraction(1, 2))
    right = RationalInterval(Fraction(2, 5), Fraction(3, 5))
    assert left * right == RationalInterval(Fraction(2, 15), Fraction(3, 10))
    assert left.positive_power(4) == RationalInterval(Fraction(1, 81), Fraction(1, 16))
    assert RationalInterval.from_payload(left.to_payload()) == left
    assert left.subset_of(RationalInterval(Fraction(0), Fraction(1)))


def test_interval_rejects_float_nan_inf_and_negative_radius() -> None:
    with pytest.raises(TypeError):
        RationalInterval(0.0, 1.0)
    with pytest.raises(TypeError):
        RationalInterval(float('nan'), float('inf'))
    with pytest.raises(ExactArithmeticError):
        RationalInterval(Fraction(2), Fraction(1))
    with pytest.raises(ExactArithmeticError):
        RationalInterval(Fraction(-1), Fraction(1)).positive_power(4)


def test_irrep_integer_convention_casimir_and_tensor_product() -> None:
    half = Irrep(1); one = Irrep(2)
    assert half.spin == Fraction(1, 2) and half.dimension == 2
    assert half.casimir == Fraction(3, 4) and one.casimir == Fraction(2)
    assert half.dual() == half and half.reverse_orientation() == half
    assert half.tensor_product(half) == (Irrep(0), Irrep(2))
    with pytest.raises((TypeError, ValueError)):
        Irrep(0.5)


In [ ]:
%%writefile tests/test_m1_coefficients.py
from __future__ import annotations

from fractions import Fraction

from src.su2_special_functions import (
    besseli_integer_positive_series_bounds, normalized_wilson_coefficient_bounds,
    wilson_character_coefficient_bounds,
)

BETA = Fraction(11, 5)


def test_bessel_positive_series_contains_higher_precision_rational_reference() -> None:
    for n in range(0, 17):
        coarse = besseli_integer_positive_series_bounds(n, BETA, 48)
        reference = besseli_integer_positive_series_bounds(n, BETA, 112)
        assert reference.subset_of(coarse)
        assert reference.lo >= 0


def test_coefficient_and_normalized_coefficient_are_positive_and_nested() -> None:
    for n in range(1, 17):
        coarse = wilson_character_coefficient_bounds(n, BETA, 56)
        fine = wilson_character_coefficient_bounds(n, BETA, 96)
        assert fine.subset_of(coarse) and fine.lo > 0
        normalized = normalized_wilson_coefficient_bounds(n, BETA, 96, 120)
        assert normalized.lo > 0 and normalized.hi <= 1


In [ ]:
%%writefile tests/test_m1_tails.py
from __future__ import annotations

from fractions import Fraction

from src.tail_bounds import gradient_tail_enclosure, value_tail_enclosure

BETA = Fraction(11, 5)
CUTOFFS = (6, 8, 10, 12, 14, 16)
PROTOTYPE_VALUE = {
    6: ('0.002677892949651685', '0.002677892949651686'),
    8: ('0.000068908927820552', '0.000068908927820553'),
    10: ('0.000001078814460529', '0.000001078814460530'),
    12: ('0.000000011303940764', '0.000000011303940765'),
    14: ('0.000000000084628276', '0.000000000084628277'),
    16: ('0.000000000000474661', '0.000000000000474662'),
}
PROTOTYPE_GRADIENT_UPPER = {
    6: '0.009776809377036234', 8: '0.000317113428056905',
    10: '0.000006015764950566', 12: '0.000000074161320899',
    14: '0.000000000638962130', 16: '0.000000000004054931',
}


def test_value_tail_nonnegative_monotone_precision_nested_and_regression() -> None:
    previous = None
    for cutoff in CUTOFFS:
        coarse = value_tail_enclosure(BETA, cutoff, 64, 80)
        fine = value_tail_enclosure(BETA, cutoff, 96, 120)
        assert fine.subset_of(coarse) and fine.lo >= 0
        if previous is not None:
            assert fine.hi <= previous.hi
        previous = fine
        fixture = RationalFixture(*PROTOTYPE_VALUE[cutoff])
        assert fine.subset_of(fixture.interval)
        assert fine.scale(Fraction(216)).hi == 216 * fine.hi


def test_gradient_tail_nonnegative_monotone_precision_nested_and_factors() -> None:
    previous = None
    for cutoff in CUTOFFS:
        coarse = gradient_tail_enclosure(BETA, cutoff, 80)
        fine = gradient_tail_enclosure(BETA, cutoff, 120)
        assert fine.subset_of(coarse) and fine.lo == 0
        if previous is not None:
            assert fine.hi <= previous.hi
        previous = fine
        assert fine.hi <= Fraction(PROTOTYPE_GRADIENT_UPPER[cutoff])
        assert fine.scale(Fraction(6)).hi == 6 * fine.hi
        assert fine.scale(Fraction(216)).hi == 216 * fine.hi


class RationalFixture:
    def __init__(self, lo: str, hi: str) -> None:
        from src.exact_arithmetic import RationalInterval
        self.interval = RationalInterval(Fraction(lo), Fraction(hi))


In [ ]:
%%writefile tests/test_m1_exact_2d_rg.py
from __future__ import annotations

from fractions import Fraction

from src.exact_2d_rg import exact_2d_rg_trajectory, normalized_fourier_ratio
from src.exact_arithmetic import RationalInterval

BETA = Fraction(11, 5)
PROTOTYPE_STEP_ZERO = {
    2: ('0.464479025270420310654070', '0.464479025270420310654071'),
    3: ('0.155492681326508526083508', '0.155492681326508526083509'),
    4: ('0.040408076198124330426319', '0.040408076198124330426320'),
}


def test_normalized_ratios_precision_nesting_and_prototype_regression() -> None:
    for n in (2, 3, 4):
        coarse = normalized_fourier_ratio(BETA, n, 64)
        fine = normalized_fourier_ratio(BETA, n, 96)
        fixture = RationalInterval(Fraction(PROTOTYPE_STEP_ZERO[n][0]), Fraction(PROTOTYPE_STEP_ZERO[n][1]))
        assert fine.subset_of(coarse) and fine.subset_of(fixture)


def test_exact_2d_interval_recurrence_is_fourth_power() -> None:
    trajectories = exact_2d_rg_trajectory(BETA, (2, 3, 4), 3, 96)
    for trajectory in trajectories.values():
        assert len(trajectory) == 4
        for before, after in zip(trajectory, trajectory[1:]):
            assert after == before.positive_power(4)
            assert after.lo >= 0 and after.hi <= before.hi


In [ ]:
%%writefile tests/test_m1_independent_verifier.py
from __future__ import annotations

from fractions import Fraction

import pytest

from src.exact_2d_rg import trajectory_payload
from src.m1_verifier import IndependentVerificationError, independent_convolution_verify

BETA = Fraction(11, 5)


def test_independent_diagonal_convolution_contains_primary() -> None:
    primary = trajectory_payload(BETA, (2, 3, 4), 3, 96, 36)
    result = independent_convolution_verify(primary, BETA, (2, 3, 4), 3, 72)
    assert result['status'] == 'PASS'
    assert result['does_not_call_primary_recurrence'] is True
    assert set(result['independent_trajectories']) == {'2', '3', '4'}
    assert len(result['checks']) == 12
    assert all(check['overlaps'] and check['primary_inside_independent'] for check in result['checks'])


def test_independent_verifier_fails_closed_on_corrupt_primary_interval() -> None:
    primary = trajectory_payload(BETA, (2, 3, 4), 3, 96, 36)
    corrupt = primary['trajectories']['2'][0]
    corrupt['lo'] = {'numerator_hex': '2', 'denominator_hex': '1'}
    corrupt['hi'] = {'numerator_hex': '2', 'denominator_hex': '1'}
    with pytest.raises(IndependentVerificationError):
        independent_convolution_verify(primary, BETA, (2, 3, 4), 3, 72)


In [ ]:
%%writefile tests/test_m1_restart.py
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path

import pytest

from src.m1_config import M1Config
from src.m1_orchestrator import _notebook_hash, create_or_resume_m1
from src.session_guard import SessionGuard, SessionState
from tests.m1_helpers import make_synthetic_accepted_parent, passing_test_report

PROJECT_ROOT = Path(__file__).resolve().parents[1]


class FakeClock:
    def __init__(self, value: float = 0.0) -> None:
        self.value = value

    def __call__(self) -> float:
        return self.value


def test_m1_notebook_identity_ignores_saved_outputs_but_tracks_source(tmp_path: Path) -> None:
    baseline = _notebook_hash(PROJECT_ROOT)
    assert baseline is not None
    source_path = PROJECT_ROOT / 'notebooks' / '20_m1_exact_2d.ipynb'
    payload = json.loads(source_path.read_text(encoding='utf-8'))
    for index, cell in enumerate(payload['cells']):
        cell['id'] = f'saved-cell-{index}'
        if cell['cell_type'] == 'code':
            cell['execution_count'] = index + 1
            cell['outputs'] = [{
                'name': 'stdout', 'output_type': 'stream', 'text': ['saved output\n'],
            }]
    project = tmp_path / 'project'
    notebook_path = project / 'notebooks' / '20_m1_exact_2d.ipynb'
    notebook_path.parent.mkdir(parents=True)
    notebook_path.write_text(json.dumps(payload, indent=1), encoding='utf-8')
    assert _notebook_hash(project) == baseline

    first_source = payload['cells'][0]['source']
    if isinstance(first_source, list):
        first_source.append('\nsource identity change')
    else:
        payload['cells'][0]['source'] = first_source + '\nsource identity change'
    notebook_path.write_text(json.dumps(payload, indent=2), encoding='utf-8')
    assert _notebook_hash(project) != baseline


def test_m1_checkpoint_resume_and_fresh_process(tmp_path: Path) -> None:
    config = make_synthetic_accepted_parent(tmp_path)
    persistent = tmp_path / 'persist'
    first = create_or_resume_m1(
        persistent, config, PROJECT_ROOT, run_id='M1-restart-test', test_report=passing_test_report(),
    )
    assert first.run_one_item_for_test() == 'M1_COEFFICIENT_BATCH'
    resumed = create_or_resume_m1(
        persistent, config, PROJECT_ROOT, run_id='M1-restart-test', test_report=passing_test_report(),
    )
    assert sum(item.status == 'done' for item in resumed.queue.items.values()) == 1
    code = (
        'from pathlib import Path; '
        'from src.m1_config import M1Config; '
        'from src.m1_orchestrator import create_or_resume_m1; '
        f'c=M1Config(parent_checkpoint_path={config.parent_checkpoint_path!r}); '
        f'o=create_or_resume_m1(Path({str(persistent)!r}),c,Path({str(PROJECT_ROOT)!r}),run_id="M1-restart-test"); '
        'assert sum(i.status=="done" for i in o.queue.items.values())==1; print(o.state.phase)'
    )
    environment = os.environ.copy()
    environment['PYTHONPATH'] = str(PROJECT_ROOT) + os.pathsep + environment.get('PYTHONPATH', '')
    completed = subprocess.run(
        [sys.executable, '-c', code], cwd=PROJECT_ROOT, env=environment,
        capture_output=True, text=True, timeout=90, check=False,
    )
    assert completed.returncode == 0, completed.stderr
    assert 'M1_BOOTSTRAP' in completed.stdout


def test_m1_session_drain_checkpoint_and_resume(tmp_path: Path) -> None:
    base = M1Config(
        checkpoint_interval_s=1.0, max_work_item_s=2.0, short_task_limit_s=0.5,
        checkpoint_reserve_s=0.1, no_long_task_after_s=3.0, drain_after_s=4.0,
        final_save_after_s=5.0, hard_return_s=6.0,
    )
    config = make_synthetic_accepted_parent(tmp_path, base)
    persistent = tmp_path / 'persist'
    orchestrator = create_or_resume_m1(persistent, config, PROJECT_ROOT, run_id='M1-drain-test')
    clock = FakeClock(); orchestrator.guard = SessionGuard(config, clock=clock)
    clock.value = base.drain_after_s
    assert orchestrator.guard.elapsed_s() == base.drain_after_s
    assert orchestrator.guard.state() is SessionState.DRAIN
    summary = orchestrator.run_until_checkpoint()
    assert 'drain checkpoint complete' in summary['stop_reason']
    resumed = create_or_resume_m1(persistent, config, PROJECT_ROOT, run_id='M1-drain-test')
    assert resumed.state.phase == 'M1_RUNNING'


def test_full_m1_run_writes_acceptance_and_remains_not_certified(tmp_path: Path) -> None:
    config = make_synthetic_accepted_parent(tmp_path)
    orchestrator = create_or_resume_m1(
        tmp_path / 'persist', config, PROJECT_ROOT, run_id='M1-full-test',
        test_report=passing_test_report(),
    )
    summary = orchestrator.run_until_checkpoint()
    assert summary['phase'] == 'M1_COMPLETE'
    assert summary['certification_status'] == 'NOT_CERTIFIED'
    assert (orchestrator.run_root / 'reports' / 'M1_report.json').is_file()
    assert (orchestrator.run_root / 'reports' / 'M1_report.md').is_file()
    assert (orchestrator.run_root / 'reports' / 'M1_acceptance.json').is_file()
    assert all(item.status == 'done' for item in orchestrator.queue.items.values())


@pytest.mark.gpu
def test_m1_optional_gpu_smoke_keeps_exact_path_on_cpu() -> None:
    torch = pytest.importorskip('torch')
    if not torch.cuda.is_available():
        pytest.skip('CUDA is unavailable.')
    from src.runtime import configure_numerics, environment_info
    configure_numerics(); tensor = torch.eye(2, dtype=torch.float64, device='cuda')
    assert float(torch.linalg.det(tensor).cpu()) == 1.0
    info = environment_info()
    assert info['cuda_available'] is True
    assert torch.backends.cuda.matmul.allow_tf32 is False
    assert M1Config().certification_status == 'NOT_CERTIFIED'


In [ ]:
%%writefile tests/test_m1_fail_closed.py
from __future__ import annotations

from pathlib import Path

import pytest

from src.checkpoint import CheckpointError, RunState
from src.common import utc_now
from src.m1_config import M1Config
from src.m1_reporting import validate_m1_acceptance
from src.work_queue import WorkQueue


def _state() -> RunState:
    config = M1Config()
    return RunState('M1-fail-closed', config.config_hash(), utc_now(), utc_now(), milestone='M1', phase='M1_RUNNING')


def test_m1_config_and_state_forbid_certified() -> None:
    with pytest.raises(ValueError):
        M1Config(certification_status='CERTIFIED')
    state = _state(); state.certification_status = 'CERTIFIED'
    with pytest.raises(CheckpointError):
        state.assert_m1_safe()


def test_missing_tail_artifact_prevents_m1_complete() -> None:
    with pytest.raises(RuntimeError, match='missing phase artifacts'):
        validate_m1_acceptance(_state(), WorkQueue(), {}, {})


def test_failed_independent_verifier_prevents_m1_complete() -> None:
    phases = (
        'M1_COEFFICIENT_BATCH', 'M1_VALUE_TAIL', 'M1_GRADIENT_TAIL',
        'M1_RG_TRAJECTORY', 'M1_INDEPENDENT_VERIFY', 'M1_REPORT',
    )
    queue = WorkQueue(); results = {}
    for phase in phases:
        item_id = queue.add(phase, 'input', {'phase': phase}, 1.0)
        item = queue.items[item_id]; item.status = 'done'
        item.result_relpath = f'{phase}.json'; item.result_sha256 = '0' * 64
        result = {}
        if phase == 'M1_COEFFICIENT_BATCH':
            result['rigor'] = 'RIGOROUS_RATIONAL_POSITIVE_SERIES'
        if phase in {'M1_VALUE_TAIL', 'M1_GRADIENT_TAIL'}:
            result['rigor'] = 'RIGOROUS_RATIONAL_ANALYTIC_BOUND'
        if phase == 'M1_RG_TRAJECTORY':
            result['rigor'] = 'RIGOROUS_RATIONAL_INTERVAL_RECURRENCE'
        if phase == 'M1_INDEPENDENT_VERIFY':
            result['status'] = 'FAIL'
        if phase == 'M1_REPORT':
            result['status'] = 'READY'
        results[phase] = {'phase': phase, 'result': result}
    tests = {
        'm0_regression_cpu_suite': 'PASS',
        'm1_required_cpu_suite': 'PASS', 'optional_gpu_suite': 'NOT_RUN_NO_CUDA',
        'm1_fresh_process_resume': 'PASS',
    }
    with pytest.raises(RuntimeError, match='independent_verifier'):
        validate_m1_acceptance(_state(), queue, results, tests)


In [ ]:
%%writefile tests/__init__.py
'''M0/M1 validation test package.'''


## M0 regression and M1 acceptance tests

The accepted M0 persistence suite is retained as a regression gate. The M1 CPU suite separately checks exact interval arithmetic, integer-valued irreps, positive-series coefficient enclosures, value/gradient-tail monotonicity and precision nesting, exact 2D recurrence, independent diagonal convolution, fail-closed acceptance, report creation, session drain, and fresh-process resume. CUDA tests are smoke tests only; all rigorous M1 enclosures are computed with CPU rational arithmetic.

In [ ]:
import json
import importlib
import sys
import time
import pytest

reload_prefixes = ('src', 'tests')
for module_name in sorted([
    name for name in sys.modules
    if any(name == prefix or name.startswith(prefix + '.') for prefix in reload_prefixes)
], reverse=True):
    del sys.modules[module_name]
importlib.invalidate_caches()

test_started = time.monotonic()
m1_test_files = [
    'tests/test_m1_exact_arithmetic.py', 'tests/test_m1_coefficients.py',
    'tests/test_m1_tails.py', 'tests/test_m1_exact_2d_rg.py',
    'tests/test_m1_independent_verifier.py', 'tests/test_m1_restart.py',
    'tests/test_m1_fail_closed.py',
]
cpu_exit = pytest.main(['-q', 'tests/test_m0.py', *m1_test_files, '-m', 'not gpu'])
if cpu_exit != pytest.ExitCode.OK:
    raise RuntimeError(f'Required M0 regression/M1 CPU tests failed with pytest exit code {int(cpu_exit)}.')
cuda_available = False
try:
    import torch
    cuda_available = torch.cuda.is_available()
except ImportError:
    cuda_available = False
gpu_status = 'NOT_RUN_NO_CUDA'
if cuda_available:
    gpu_exit = pytest.main(['-q', 'tests/test_m0.py', 'tests/test_m1_restart.py', '-m', 'gpu'])
    if gpu_exit != pytest.ExitCode.OK:
        raise RuntimeError(f'CUDA is present but the M0/M1 GPU smoke suite failed: {int(gpu_exit)}.')
    gpu_status = 'PASS'
M1_TEST_REPORT = {
    'm0_regression_cpu_suite': 'PASS',
    'm1_required_cpu_suite': 'PASS',
    'optional_gpu_suite': gpu_status,
    'm1_fresh_process_resume': 'PASS',
    'elapsed_s': time.monotonic() - test_started,
}
print(json.dumps(M1_TEST_REPORT, indent=2))

## Create or resume a new durable M1 run

This cell never resumes the accepted M0 run. It verifies M0 checkpoint `ckpt_000014` read-only, fixes its `hashes.json` digest and the acceptance-record digest into a new `M1-...` manifest, and then creates or resumes only `VALIDATED_RG_M1_RUN_ID`.

In [ ]:
import importlib
import json
import sys

# Clear generated modules when this bootstrap cell is deliberately rerun in one kernel.
for module_name in sorted([name for name in sys.modules if name == 'src' or name.startswith('src.')], reverse=True):
    del sys.modules[module_name]
importlib.invalidate_caches()

from src.m1_config import M1Config
from src.m1_orchestrator import create_or_resume_m1
from src.runtime import environment_info, validate_persistent_root

M1_CONFIG = M1Config()
PERSIST_ROOT = validate_persistent_root()
RUNTIME = environment_info()
print(json.dumps(RUNTIME, ensure_ascii=False, indent=2))
print('M1 config hash:', M1_CONFIG.config_hash())
print('Persistent root:', PERSIST_ROOT)
m1_orchestrator = create_or_resume_m1(
    PERSIST_ROOT, M1_CONFIG, PROJECT_ROOT,
    run_id=os.environ.get('VALIDATED_RG_M1_RUN_ID'),
    test_report=M1_TEST_REPORT,
)
assert m1_orchestrator.state.milestone == 'M1'
assert m1_orchestrator.state.certification_status == 'NOT_CERTIFIED'

## Single bounded-session entry point

The runner starts no item whose recorded prediction violates the p95/reserve rule, checkpoints before and after each item and every 15 minutes, begins final checkpointing by the drain/final-save thresholds, verifies the committed checkpoint immediately, and returns with exact next-session instructions.

In [ ]:
result = m1_orchestrator.run_until_checkpoint()
assert result['certification_status'] == 'NOT_CERTIFIED'
result

## Inspect the latest valid M1 checkpoint and report

Inspection is read-only. It revalidates the newest M1 checkpoint and prints M1 report/acceptance data plus the accepted-M0 parent identity.

In [ ]:
import json

loaded = m1_orchestrator.checkpoints.load_latest(restore_rng=False)
if loaded is None:
    raise RuntimeError('No valid committed checkpoint exists after the run.')
inspection = {
    'checkpoint': str(loaded.path),
    'checkpoint_index': loaded.state.checkpoint_index,
    'phase': loaded.state.phase,
    'certification_status': loaded.state.certification_status,
    'done': sum(item.status == 'done' for item in loaded.queue.items.values()),
    'pending': sum(item.status == 'pending' for item in loaded.queue.items.values()),
    'skipped_invalid': list(loaded.skipped_invalid),
}
print(json.dumps(inspection, ensure_ascii=False, indent=2))
report_path = m1_orchestrator.run_root / 'reports' / 'M1_report.json'
if report_path.is_file():
    print(json.dumps(json.loads(report_path.read_text(encoding='utf-8')), ensure_ascii=False, indent=2))
else:
    print('M1 report is written only after all M1 acceptance gates pass.')
manifest = json.loads((m1_orchestrator.run_root / 'run_manifest.json').read_text(encoding='utf-8'))
print('Accepted M0 parent:')
print(json.dumps({key: manifest[key] for key in (
    'parent_milestone', 'parent_run_id', 'parent_checkpoint', 'parent_checkpoint_path',
    'parent_checkpoint_hash_manifest_sha256', 'm0_acceptance_record_sha256',
)}, ensure_ascii=False, indent=2))
print('Governing document hashes:')
print(json.dumps(manifest['governing_document_hashes'], ensure_ascii=False, indent=2))
print('Full-plan reference artifact hashes:')
print(json.dumps(manifest['reference_artifact_hashes'], ensure_ascii=False, indent=2))

## Scope boundary after M1

The notebook stops here for milestone review. M1 proves only the listed rational Wilson-coefficient enclosures, representation value/gradient tails, and finite exact 2D trajectories. The 4D armillary construction, Triad-ATRG, forward derivatives, deterministic residual validation, influence matrices, and mass-gap/continuum bridges remain outside this artifact.

## Governing M1–M6 full plan

The version-0.2 full plan is immutable, SHA-256-recorded run governance. The supplied prototype/report are used only as displayed regression fixtures; all current endpoints are recomputed from exact positive rational series.

| Milestone | Required outcome before advancing | Status in this artifact |
|---|---|---|
| M0 | persistence, recovery, bounded session, external execution tests | accepted parent; frozen |
| M1 | exact 2D recurrence and rigorous value/gradient tails | implemented; external execution acceptance pending |
| M2 | low-cutoff dense/armillary equivalence | not started |
| M3 | matrix-free GPU Triad-ATRG pilot | not started |
| M4 | forward source tangents and complete error DAG | not started |
| M5 | deterministic one-step enclosure, P1–P11 gates | not started |
| M6 | multi-step balls, P12–P13, independent final verifier | not started |

`CERTIFIED` is forbidden unless P0–P13 all pass and an independent verifier proves `q_cert_upper < 1`. After external execution, review `M1_report.json`, `M1_acceptance.json`, and session artifacts. M2 must not begin until M1 is separately accepted.